# QGECAF A0–A9 ablation study using the same locked patients

This notebook is the companion to `resumeQGECAF_CleanV2_FromScratch_Resumable_FIXED.ipynb`. It does **not** create new outer folds. It reads `QGECAF_FROM_SCRATCH_RUN_01/locked_patient_fold_assignments.csv` and uses the same 11,265 development images and the same ten patient-level outer holdout folds for A0–A9.

The 1,135-image, 180-patient final test cohort (`outer_fold = -1`) is audited but is never used for fitting, validation, model selection, preprocessing, or ablation evaluation.

## Ablation variants

| ID | Model | Purpose |
|---|---|---|
| A0 | MobileNetV2 deep stream only | CNN baseline |
| A1 | Handcrafted stream only | Independent handcrafted contribution |
| A2 | Deep + handcrafted concatenation | Naïve fusion baseline |
| A3 | Deep-query attention only | Deep tokens query handcrafted tokens |
| A4 | Handcrafted-query attention only | Handcrafted tokens query deep tokens |
| A5 | Bidirectional attention + fixed averaging | Removes adaptive gating |
| A6 | Bidirectional attention + learned global gate | Static learned fusion weight |
| A7 | Quality-gated fusion + evidential loss without KL | Tests KL regularisation |
| A8 | Quality-gated fusion + softmax | Non-evidential full-fusion control |
| A9 | Full QGECAF | Quality gate + evidential loss + KL |

Run this notebook in a fresh `chaitalienv` kernel. Execute the cells in order. Re-running the training cell is safe: completed variant-folds are skipped, while an interrupted active fit resumes from the last saved epoch.


## 1. Imports, configuration classes and resumable callbacks


In [1]:
"""Patient-level ten-fold ablation study for QGECAF.

This runner reuses the data discovery, feature extraction, group-aware BGWO,
and leakage-audit functions from the publication-ready QGECAF notebook/script.
It trains A0--A9 on the same locked outer folds and saves fold-level training
time, epoch time, inference time, accuracy, precision, recall, F1, Cohen's
kappa, MCC, calibration metrics, predictions, and publication summaries.

Recommended Windows command (Anaconda Prompt):

    conda activate chaitalienv
    cd /d D:\\chaitali\\qgacef
    python QGECAF_Ablation_Study.py ^
      --base-code resumeQGECAF_CleanV2_FromScratch_Resumable_FIXED.ipynb ^
      --dataset C:\\Users\\viraj\\Downloads\\patientsplit801010 ^
      --metadata-csv C:\\Users\\viraj\\Downloads\\patientsplit801010\\patient_split_80_10_10.csv ^
      --source-results D:\\chaitali\\qgacef\\QGECAF_FROM_SCRATCH_RUN_01 ^
      --output D:\\chaitali\\qgacef\\QGECAF_ABLATION_SAME_LOCKED_PATIENTS

Run a subset first, if desired:

    python QGECAF_Ablation_Study.py --variants A0,A2,A5,A8,A9 [other arguments]

Resume is enabled by default. Re-run the same command after an interruption;
completed variant-folds are skipped and an active variant-fold continues from
its latest completed epoch. Use --no-resume only to restart requested runs.

The untouched test cohort is deliberately not evaluated by this script.
Ablation selection must use only the development cross-validation results.
"""

from __future__ import annotations

import argparse
import gc
import importlib.util
import json
import math
import os
import random
import shutil
import sys
import time
import types
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Sequence, Tuple

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.stats import friedmanchisquare, t, wilcoxon
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)


VARIANT_INFO = {
    "A0": "MobileNetV2 deep stream only",
    "A1": "Handcrafted stream only",
    "A2": "Deep and handcrafted concatenation",
    "A3": "Deep-query cross-attention only",
    "A4": "Handcrafted-query cross-attention only",
    "A5": "Bidirectional attention with fixed averaging",
    "A6": "Bidirectional attention with learned global gate",
    "A7": "Quality-gated fusion, evidential loss without KL",
    "A8": "Quality-gated fusion with softmax head",
    "A9": "Full QGECAF: quality gate and evidential loss with KL",
}

EVIDENTIAL_VARIANTS = {"A0", "A1", "A2", "A3", "A4", "A5", "A6", "A7", "A9"}
QUALITY_GATE_VARIANTS = {"A7", "A8", "A9"}


@dataclass
class RunConfig:
    base_code: str = "resumeQGECAF_CleanV2_FromScratch_Resumable_FIXED.ipynb"
    dataset: str = r"D:\chaitali\patientsplit801010_clean_v2"
    metadata_csv: str = (
        r"D:\chaitali\patientsplit801010_clean_v2\patient_split_80_10_10.csv"
    )
    source_results: str = (
        r"D:\chaitali\qgacef\QGECAF_FROM_SCRATCH_RUN_01"
    )
    output: str = (
        r"D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS"
    )
    patient_column: str = "Patient_num"
    image_column: str = "Image_name"
    variants: str = "A0,A1,A2,A3,A4,A5,A6,A7,A8,A9"
    image_size: int = 128
    seed: int = 42
    pca_variance: float = 0.98
    bgwo_agents: int = 12
    bgwo_iters: int = 25
    bgwo_cv: int = 3
    bgwo_alpha: float = 0.99
    embed_dim: int = 64
    attention_heads: int = 4
    attention_dropout: float = 0.10
    batch_size: int = 8
    inference_batch_size: int = 16
    epochs: int = 50
    learning_rate: float = 1e-4
    kl_weight: float = 1e-3
    dropout: float = 0.30
    fine_tune_last: int = 30
    outer_folds: int = 10
    inner_folds: int = 9
    early_stopping_patience: int = 8
    reduce_lr_patience: int = 3
    resume: bool = True
    force_features: bool = False


def parse_args() -> RunConfig:
    defaults = RunConfig()
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--base-code", default=defaults.base_code)
    parser.add_argument("--dataset", default=defaults.dataset)
    parser.add_argument("--metadata-csv", default=defaults.metadata_csv)
    parser.add_argument("--source-results", default=defaults.source_results)
    parser.add_argument("--output", default=defaults.output)
    parser.add_argument("--patient-column", default=defaults.patient_column)
    parser.add_argument("--image-column", default=defaults.image_column)
    parser.add_argument("--variants", default=defaults.variants)
    parser.add_argument("--image-size", type=int, default=defaults.image_size)
    parser.add_argument("--seed", type=int, default=defaults.seed)
    parser.add_argument("--pca-variance", type=float, default=defaults.pca_variance)
    parser.add_argument("--bgwo-agents", type=int, default=defaults.bgwo_agents)
    parser.add_argument("--bgwo-iters", type=int, default=defaults.bgwo_iters)
    parser.add_argument("--bgwo-cv", type=int, default=defaults.bgwo_cv)
    parser.add_argument("--bgwo-alpha", type=float, default=defaults.bgwo_alpha)
    parser.add_argument("--embed-dim", type=int, default=defaults.embed_dim)
    parser.add_argument("--attention-heads", type=int, default=defaults.attention_heads)
    parser.add_argument("--batch-size", type=int, default=defaults.batch_size)
    parser.add_argument(
        "--inference-batch-size", type=int, default=defaults.inference_batch_size
    )
    parser.add_argument("--epochs", type=int, default=defaults.epochs)
    parser.add_argument("--learning-rate", type=float, default=defaults.learning_rate)
    parser.add_argument("--kl-weight", type=float, default=defaults.kl_weight)
    parser.add_argument("--dropout", type=float, default=defaults.dropout)
    parser.add_argument("--fine-tune-last", type=int, default=defaults.fine_tune_last)
    parser.add_argument("--outer-folds", type=int, default=defaults.outer_folds)
    parser.add_argument("--inner-folds", type=int, default=defaults.inner_folds)
    parser.add_argument(
        "--early-stopping-patience",
        type=int,
        default=defaults.early_stopping_patience,
    )
    parser.add_argument("--no-resume", action="store_true")
    parser.add_argument("--force-features", action="store_true")
    args = parser.parse_args()
    values = vars(args)
    values["resume"] = not values.pop("no_resume")
    return RunConfig(**values)


def load_publication_module(path: str | Path) -> types.ModuleType:
    """Load definitions from either the publication .py file or its notebook."""
    source_path = Path(path).resolve()
    if not source_path.exists():
        raise FileNotFoundError(
            f"Base QGECAF code was not found: {source_path}\n"
            "Pass --base-code with the publication-ready .ipynb or .py path."
        )

    cache = globals().setdefault("_QGECAF_BASE_MODULE_CACHE", {})
    cache_key = str(source_path)
    if cache_key in cache:
        return cache[cache_key]

    module = types.ModuleType("qgecaf_publication_base")
    module.__file__ = str(source_path)
    sys.modules[module.__name__] = module

    if source_path.suffix.lower() == ".py":
        spec = importlib.util.spec_from_file_location(module.__name__, source_path)
        if spec is None or spec.loader is None:
            raise ImportError(f"Unable to load {source_path}")
        module = importlib.util.module_from_spec(spec)
        sys.modules[module.__name__] = module
        spec.loader.exec_module(module)
        cache[cache_key] = module
        return module

    if source_path.suffix.lower() != ".ipynb":
        raise ValueError("--base-code must refer to a .py or .ipynb file.")

    notebook = json.loads(source_path.read_text(encoding="utf-8"))
    candidates = []
    required_markers = (
        "def discover_presplit_dataset",
        "def fit_preprocessing",
        "class EvidentialLoss",
        "def choose_inner_split",
    )
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        if all(marker in source for marker in required_markers):
            candidates.append(source)
    if not candidates:
        raise RuntimeError(
            "The notebook does not contain the complete publication-ready definitions cell."
        )
    source = max(candidates, key=len)
    exec(compile(source, str(source_path), "exec"), module.__dict__)
    cache[cache_key] = module
    return module


def set_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except (AttributeError, RuntimeError):
        pass


class GlobalConvexFusion(tf.keras.layers.Layer):
    """A single learned gate shared by every sample."""

    def build(self, input_shape):
        self.gate_logit = self.add_weight(
            name="gate_logit", shape=(), initializer="zeros", trainable=True
        )
        super().build(input_shape)

    def call(self, inputs):
        deep_vector, hand_vector = inputs
        gate = tf.math.sigmoid(self.gate_logit)
        return gate * deep_vector + (1.0 - gate) * hand_vector


def write_json_atomic(path: str | Path, payload: Mapping) -> None:
    """Write JSON without leaving a partially written state file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    os.replace(temporary, path)


def read_deduplicated_csv(path: str | Path, epoch_column: str = "epoch") -> pd.DataFrame:
    """Read a resumable CSV log and retain the latest copy of each epoch."""
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    frame = pd.read_csv(path)
    if epoch_column in frame.columns:
        frame = (
            frame.drop_duplicates(subset=[epoch_column], keep="last")
            .sort_values(epoch_column)
            .reset_index(drop=True)
        )
    return frame


class PersistentEpochTimer(tf.keras.callbacks.Callback):
    """Persist each completed epoch immediately so timing survives interruption."""

    def __init__(self, csv_path: str | Path):
        super().__init__()
        self.csv_path = Path(csv_path)
        previous = read_deduplicated_csv(self.csv_path)
        self.rows: List[dict] = previous.to_dict("records")
        self._start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._start = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        row = {
            "epoch": int(epoch + 1),
            "epoch_time_seconds": float(time.perf_counter() - self._start),
        }
        row.update({key: float(value) for key, value in (logs or {}).items()})
        self.rows = [
            previous for previous in self.rows
            if int(previous["epoch"]) != row["epoch"]
        ]
        self.rows.append(row)
        frame = pd.DataFrame(self.rows).sort_values("epoch").reset_index(drop=True)
        temporary = self.csv_path.with_name(self.csv_path.name + ".tmp")
        frame.to_csv(temporary, index=False)
        os.replace(temporary, self.csv_path)

    def frame(self) -> pd.DataFrame:
        return pd.DataFrame(self.rows).sort_values("epoch").reset_index(drop=True)


class PersistentModelCheckpoint(tf.keras.callbacks.ModelCheckpoint):
    """ModelCheckpoint whose best monitored value survives process restarts."""

    def __init__(self, *args, state_path: str | Path, **kwargs):
        self.state_path = Path(state_path)
        super().__init__(*args, **kwargs)

    def on_train_begin(self, logs=None):
        super().on_train_begin(logs)
        if self.state_path.exists():
            state = json.loads(self.state_path.read_text(encoding="utf-8"))
            if state.get("best") is not None:
                self.best = float(state["best"])

    def on_epoch_end(self, epoch, logs=None):
        super().on_epoch_end(epoch, logs)
        best = float(self.best)
        write_json_atomic(
            self.state_path,
            {"best": best if np.isfinite(best) else None},
        )


class PersistentEarlyStopping(tf.keras.callbacks.EarlyStopping):
    """EarlyStopping with persistent best/wait counters."""

    def __init__(self, *args, state_path: str | Path, **kwargs):
        self.state_path = Path(state_path)
        super().__init__(*args, **kwargs)

    def on_train_begin(self, logs=None):
        super().on_train_begin(logs)
        if self.state_path.exists():
            state = json.loads(self.state_path.read_text(encoding="utf-8"))
            if state.get("best") is not None:
                self.best = float(state["best"])
            self.wait = int(state.get("wait", 0))
            self.stopped_epoch = int(state.get("stopped_epoch", 0))
            if hasattr(self, "best_epoch"):
                self.best_epoch = int(state.get("best_epoch", 0))

    def on_epoch_end(self, epoch, logs=None):
        super().on_epoch_end(epoch, logs)
        best = float(self.best)
        state = {
            "best": best if np.isfinite(best) else None,
            "wait": int(self.wait),
            "stopped_epoch": int(self.stopped_epoch),
        }
        if hasattr(self, "best_epoch"):
            state["best_epoch"] = int(self.best_epoch)
        write_json_atomic(self.state_path, state)


class PersistentReduceLROnPlateau(tf.keras.callbacks.ReduceLROnPlateau):
    """ReduceLROnPlateau with persistent patience and cooldown state."""

    def __init__(self, *args, state_path: str | Path, **kwargs):
        self.state_path = Path(state_path)
        super().__init__(*args, **kwargs)

    def on_train_begin(self, logs=None):
        super().on_train_begin(logs)
        if self.state_path.exists():
            state = json.loads(self.state_path.read_text(encoding="utf-8"))
            if state.get("best") is not None:
                self.best = float(state["best"])
            self.wait = int(state.get("wait", 0))
            self.cooldown_counter = int(state.get("cooldown_counter", 0))

    def on_epoch_end(self, epoch, logs=None):
        super().on_epoch_end(epoch, logs)
        best = float(self.best)
        write_json_atomic(
            self.state_path,
            {
                "best": best if np.isfinite(best) else None,
                "wait": int(self.wait),
                "cooldown_counter": int(self.cooldown_counter),
            },
        )


## 2. Model streams, attention mechanisms and A0–A9 variants


In [2]:
def build_deep_tokens(cfg: RunConfig):
    image_input = tf.keras.layers.Input(
        (cfg.image_size, cfg.image_size, 1), name="image"
    )
    rgb = tf.keras.layers.Concatenate(name="grayscale_to_rgb")(
        [image_input, image_input, image_input]
    )
    rgb = tf.keras.layers.Rescaling(
        scale=2.0, offset=-1.0, name="mobilenet_preprocess"
    )(rgb)
    backbone = tf.keras.applications.MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(cfg.image_size, cfg.image_size, 3),
    )
    backbone.trainable = True
    freeze_until = max(0, len(backbone.layers) - cfg.fine_tune_last)
    for layer in backbone.layers[:freeze_until]:
        layer.trainable = False
    feature_map = backbone(rgb)
    projected = tf.keras.layers.Conv2D(
        cfg.embed_dim, 1, padding="same", name="deep_projection"
    )(feature_map)
    height, width = projected.shape[1], projected.shape[2]
    if height is None or width is None:
        raise ValueError("The MobileNetV2 spatial dimensions must be static.")
    tokens = tf.keras.layers.Reshape(
        (int(height * width), cfg.embed_dim), name="deep_spatial_tokens"
    )(projected)
    tokens = tf.keras.layers.LayerNormalization(name="deep_token_norm")(tokens)
    return image_input, tokens


def build_handcrafted_tokens(feature_dims: Mapping[str, int], cfg: RunConfig):
    inputs, tokens = [], []
    for family in ("wst", "gabor", "lbp", "hog", "shape"):
        family_input = tf.keras.layers.Input(
            (int(feature_dims[family]),), name=family
        )
        token = tf.keras.layers.Dense(
            cfg.embed_dim, activation="gelu", name=f"{family}_token_projection"
        )(family_input)
        token = tf.keras.layers.LayerNormalization(
            name=f"{family}_token_norm"
        )(token)
        token = tf.keras.layers.Reshape(
            (1, cfg.embed_dim), name=f"{family}_token"
        )(token)
        inputs.append(family_input)
        tokens.append(token)
    token_bank = tf.keras.layers.Concatenate(
        axis=1, name="handcrafted_family_tokens"
    )(tokens)
    return inputs, token_bank


def cross_attention_vectors(deep_tokens, hand_tokens, cfg: RunConfig):
    key_dim = cfg.embed_dim // cfg.attention_heads
    deep_update = tf.keras.layers.MultiHeadAttention(
        num_heads=cfg.attention_heads,
        key_dim=key_dim,
        dropout=cfg.attention_dropout,
        name="deep_query_attention",
    )(query=deep_tokens, key=hand_tokens, value=hand_tokens)
    deep_enriched = tf.keras.layers.Add(name="deep_attention_residual")(
        [deep_tokens, deep_update]
    )
    deep_enriched = tf.keras.layers.LayerNormalization(
        name="deep_attention_norm"
    )(deep_enriched)

    hand_update = tf.keras.layers.MultiHeadAttention(
        num_heads=cfg.attention_heads,
        key_dim=key_dim,
        dropout=cfg.attention_dropout,
        name="handcrafted_query_attention",
    )(query=hand_tokens, key=deep_tokens, value=deep_tokens)
    hand_enriched = tf.keras.layers.Add(name="handcrafted_attention_residual")(
        [hand_tokens, hand_update]
    )
    hand_enriched = tf.keras.layers.LayerNormalization(
        name="handcrafted_attention_norm"
    )(hand_enriched)

    deep_vector = tf.keras.layers.GlobalAveragePooling1D(
        name="deep_cross_attention_pool"
    )(deep_enriched)
    hand_vector = tf.keras.layers.GlobalAveragePooling1D(
        name="handcrafted_cross_attention_pool"
    )(hand_enriched)
    return deep_vector, hand_vector


def build_ablation_model(
    variant: str,
    feature_dims: Mapping[str, int],
    num_classes: int,
    cfg: RunConfig,
) -> tf.keras.Model:
    if variant not in VARIANT_INFO:
        raise ValueError(f"Unknown ablation variant: {variant}")

    needs_deep = variant != "A1"
    needs_hand = variant != "A0"
    model_inputs = []
    deep_tokens = hand_tokens = None

    if needs_deep:
        image_input, deep_tokens = build_deep_tokens(cfg)
        model_inputs.append(image_input)
    if needs_hand:
        family_inputs, hand_tokens = build_handcrafted_tokens(feature_dims, cfg)
        model_inputs.extend(family_inputs)

    if variant == "A0":
        fused = tf.keras.layers.GlobalAveragePooling1D(name="deep_only_pool")(
            deep_tokens
        )
    elif variant == "A1":
        fused = tf.keras.layers.GlobalAveragePooling1D(name="handcrafted_only_pool")(
            hand_tokens
        )
    elif variant == "A2":
        deep_raw = tf.keras.layers.GlobalAveragePooling1D(name="deep_raw_pool")(
            deep_tokens
        )
        hand_raw = tf.keras.layers.GlobalAveragePooling1D(name="handcrafted_raw_pool")(
            hand_tokens
        )
        fused = tf.keras.layers.Concatenate(name="naive_concatenation")(
            [deep_raw, hand_raw]
        )
        fused = tf.keras.layers.Dense(
            cfg.embed_dim, activation="gelu", name="concat_projection"
        )(fused)
        fused = tf.keras.layers.LayerNormalization(name="concat_projection_norm")(
            fused
        )
    elif variant == "A3":
        key_dim = cfg.embed_dim // cfg.attention_heads
        update = tf.keras.layers.MultiHeadAttention(
            num_heads=cfg.attention_heads,
            key_dim=key_dim,
            dropout=cfg.attention_dropout,
            name="deep_query_attention",
        )(query=deep_tokens, key=hand_tokens, value=hand_tokens)
        enriched = tf.keras.layers.LayerNormalization(name="deep_attention_norm")(
            tf.keras.layers.Add(name="deep_attention_residual")(
                [deep_tokens, update]
            )
        )
        fused = tf.keras.layers.GlobalAveragePooling1D(
            name="deep_cross_attention_pool"
        )(enriched)
    elif variant == "A4":
        key_dim = cfg.embed_dim // cfg.attention_heads
        update = tf.keras.layers.MultiHeadAttention(
            num_heads=cfg.attention_heads,
            key_dim=key_dim,
            dropout=cfg.attention_dropout,
            name="handcrafted_query_attention",
        )(query=hand_tokens, key=deep_tokens, value=deep_tokens)
        enriched = tf.keras.layers.LayerNormalization(
            name="handcrafted_attention_norm"
        )(
            tf.keras.layers.Add(name="handcrafted_attention_residual")(
                [hand_tokens, update]
            )
        )
        fused = tf.keras.layers.GlobalAveragePooling1D(
            name="handcrafted_cross_attention_pool"
        )(enriched)
    else:
        deep_vector, hand_vector = cross_attention_vectors(
            deep_tokens, hand_tokens, cfg
        )
        if variant == "A5":
            fused = tf.keras.layers.Average(name="fixed_average_fusion")(
                [deep_vector, hand_vector]
            )
        elif variant == "A6":
            fused = GlobalConvexFusion(name="global_gate_fusion")(
                [deep_vector, hand_vector]
            )
        else:
            quality_input = tf.keras.layers.Input((6,), name="quality")
            model_inputs.append(quality_input)
            gate_hidden = tf.keras.layers.Dense(
                16, activation="relu", name="quality_gate_hidden"
            )(quality_input)
            gate = tf.keras.layers.Dense(
                1, activation="sigmoid", name="quality_gate_lambda"
            )(gate_hidden)
            inverse_gate = tf.keras.layers.Lambda(
                lambda value: 1.0 - value, name="inverse_quality_gate"
            )(gate)
            gated_deep = tf.keras.layers.Multiply(name="gated_deep")(
                [deep_vector, gate]
            )
            gated_hand = tf.keras.layers.Multiply(name="gated_handcrafted")(
                [hand_vector, inverse_gate]
            )
            fused = tf.keras.layers.Add(name="quality_gated_fusion")(
                [gated_deep, gated_hand]
            )

    fused = tf.keras.layers.LayerNormalization(name="fusion_norm")(fused)
    fused = tf.keras.layers.Dense(
        128, activation="gelu", name="fusion_dense"
    )(fused)
    fused = tf.keras.layers.Dropout(cfg.dropout, name="fusion_dropout")(fused)

    if variant == "A8":
        output = tf.keras.layers.Dense(
            num_classes, activation="softmax", name="class_probability"
        )(fused)
    else:
        evidence = tf.keras.layers.Dense(
            num_classes, activation="softplus", name="class_evidence"
        )(fused)
        output = tf.keras.layers.Lambda(
            lambda value: value + 1.0, name="dirichlet_alpha"
        )(evidence)

    return tf.keras.Model(
        inputs=model_inputs,
        outputs=output,
        name=f"QGECAF_Ablation_{variant}",
    )


def weighted_categorical_crossentropy(class_weights: np.ndarray):
    weights = tf.constant(class_weights, dtype=tf.float32)

    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, y_pred.dtype)
        y_pred = tf.clip_by_value(y_pred, 1e-8, 1.0)
        sample_weights = tf.reduce_sum(y_true * tf.cast(weights, y_pred.dtype), axis=1)
        categorical_ce = -tf.reduce_sum(y_true * tf.math.log(y_pred), axis=1)
        return sample_weights * categorical_ce

    return loss


def compile_ablation_model(
    model: tf.keras.Model,
    variant: str,
    y_train: np.ndarray,
    num_classes: int,
    cfg: RunConfig,
    base,
) -> None:
    weights = base.compute_class_weights(y_train, num_classes)
    optimizer = tf.keras.optimizers.Adam(learning_rate=cfg.learning_rate)
    if variant == "A8":
        model.compile(
            optimizer=optimizer,
            loss=weighted_categorical_crossentropy(weights),
            metrics=[tf.keras.metrics.CategoricalAccuracy(name="accuracy")],
        )
    else:
        kl_weight = 0.0 if variant == "A7" else cfg.kl_weight
        model.compile(
            optimizer=optimizer,
            loss=base.EvidentialLoss(num_classes, kl_weight, weights),
            metrics=[base.DirichletMeanAccuracy()],
        )


def required_input_names(model: tf.keras.Model) -> set:
    return {tensor.name.split(":", 1)[0] for tensor in model.inputs}


def select_inputs(
    all_inputs: Mapping[str, np.ndarray], model: tf.keras.Model
) -> Dict[str, np.ndarray]:
    required = required_input_names(model)
    missing = required.difference(all_inputs)
    if missing:
        raise KeyError(f"Missing model inputs: {sorted(missing)}")
    return {name: all_inputs[name] for name in required}


def slice_inputs(inputs: Mapping[str, np.ndarray], count: int):
    return {name: values[:count] for name, values in inputs.items()}


## 3. Holdout evaluation, dataset validation and statistical analysis


In [3]:
def expected_calibration_error(
    y_true: np.ndarray, probabilities: np.ndarray, bins: int = 15
) -> float:
    confidence = probabilities.max(axis=1)
    prediction = probabilities.argmax(axis=1)
    correct = prediction == y_true
    edges = np.linspace(0.0, 1.0, bins + 1)
    result = 0.0
    for lower, upper in zip(edges[:-1], edges[1:]):
        selected = (confidence > lower) & (confidence <= upper)
        if selected.any():
            result += selected.mean() * abs(
                correct[selected].mean() - confidence[selected].mean()
            )
    return float(result)



def save_training_curves(history: pd.DataFrame, output_dir: Path) -> None:
    """Save loss and accuracy curves from a completed or resumed fit."""
    if history is None or history.empty:
        return
    output_dir.mkdir(parents=True, exist_ok=True)
    epochs = pd.to_numeric(history["epoch"], errors="coerce") + 1

    if {"loss", "val_loss"}.issubset(history.columns):
        fig, axis = plt.subplots(figsize=(8, 5))
        axis.plot(epochs, history["loss"], label="Training loss")
        axis.plot(epochs, history["val_loss"], label="Validation loss")
        axis.set_xlabel("Epoch")
        axis.set_ylabel("Loss")
        axis.legend()
        axis.grid(alpha=0.2)
        fig.tight_layout()
        fig.savefig(output_dir / "training_loss.png", dpi=300, bbox_inches="tight")
        plt.close(fig)

    training_accuracy = next(
        (
            column
            for column in history.columns
            if "accuracy" in column.casefold()
            and not column.casefold().startswith("val_")
        ),
        None,
    )
    validation_accuracy = (
        f"val_{training_accuracy}" if training_accuracy is not None else None
    )
    if (
        training_accuracy is not None
        and validation_accuracy in history.columns
    ):
        fig, axis = plt.subplots(figsize=(8, 5))
        axis.plot(epochs, history[training_accuracy], label="Training accuracy")
        axis.plot(epochs, history[validation_accuracy], label="Validation accuracy")
        axis.set_xlabel("Epoch")
        axis.set_ylabel("Accuracy")
        axis.legend()
        axis.grid(alpha=0.2)
        fig.tight_layout()
        fig.savefig(output_dir / "training_accuracy.png", dpi=300, bbox_inches="tight")
        plt.close(fig)


def save_evaluation_plots(
    y_true: np.ndarray,
    prediction: np.ndarray,
    probabilities: np.ndarray,
    class_names: Sequence[str],
    output_dir: Path,
) -> None:
    """Save a confusion matrix and 15-bin reliability diagram."""
    output_dir.mkdir(parents=True, exist_ok=True)
    matrix = confusion_matrix(
        y_true, prediction, labels=np.arange(len(class_names))
    )
    fig, axis = plt.subplots(figsize=(9, 8))
    image = axis.imshow(matrix, interpolation="nearest", cmap="Blues")
    fig.colorbar(image, ax=axis)
    axis.set(
        xticks=np.arange(len(class_names)),
        yticks=np.arange(len(class_names)),
        xticklabels=class_names,
        yticklabels=class_names,
        xlabel="Predicted class",
        ylabel="True class",
    )
    plt.setp(axis.get_xticklabels(), rotation=45, ha="right")
    threshold = matrix.max() / 2.0 if matrix.size else 0.0
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            axis.text(
                column,
                row,
                str(matrix[row, column]),
                ha="center",
                va="center",
                color="white" if matrix[row, column] > threshold else "black",
            )
    fig.tight_layout()
    fig.savefig(output_dir / "confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    confidence = probabilities.max(axis=1)
    correct = (prediction == y_true).astype(float)
    edges = np.linspace(0.0, 1.0, 16)
    mean_confidence, empirical_accuracy = [], []
    for index in range(15):
        if index == 14:
            mask = (confidence >= edges[index]) & (confidence <= edges[index + 1])
        else:
            mask = (confidence >= edges[index]) & (confidence < edges[index + 1])
        if np.any(mask):
            mean_confidence.append(float(confidence[mask].mean()))
            empirical_accuracy.append(float(correct[mask].mean()))
    fig, axis = plt.subplots(figsize=(7, 7))
    axis.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
    axis.plot(mean_confidence, empirical_accuracy, "o-", label="Model")
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1.05)
    axis.set_xlabel("Mean confidence")
    axis.set_ylabel("Empirical accuracy")
    axis.set_title("Reliability diagram")
    axis.legend()
    axis.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(output_dir / "reliability_diagram.png", dpi=300, bbox_inches="tight")
    plt.close(fig)


def timed_evaluation(
    model: tf.keras.Model,
    variant: str,
    inputs: Mapping[str, np.ndarray],
    y_true: np.ndarray,
    class_names: Sequence[str],
    paths: np.ndarray,
    output_dir: Path,
    cfg: RunConfig,
) -> dict:
    warmup_count = min(cfg.inference_batch_size, len(y_true))
    model.predict(
        slice_inputs(inputs, warmup_count),
        batch_size=cfg.inference_batch_size,
        verbose=0,
    )
    start = time.perf_counter()
    raw_output = model.predict(
        inputs, batch_size=cfg.inference_batch_size, verbose=0
    )
    inference_seconds = float(time.perf_counter() - start)

    if variant in EVIDENTIAL_VARIANTS:
        alpha = np.asarray(raw_output, dtype=np.float64)
        strength = alpha.sum(axis=1, keepdims=True)
        probabilities = alpha / strength
        uncertainty = len(class_names) / strength[:, 0]
    else:
        alpha = None
        probabilities = np.asarray(raw_output, dtype=np.float64)
        probabilities /= np.maximum(probabilities.sum(axis=1, keepdims=True), 1e-12)
        uncertainty = -np.sum(
            probabilities * np.log(np.clip(probabilities, 1e-12, 1.0)), axis=1
        ) / math.log(len(class_names))

    prediction = probabilities.argmax(axis=1)
    one_hot = tf.keras.utils.to_categorical(y_true, len(class_names))
    incorrect = (prediction != y_true).astype(np.int32)
    try:
        uncertainty_auc = float(roc_auc_score(incorrect, uncertainty))
    except ValueError:
        uncertainty_auc = float("nan")

    nll = float(
        -np.mean(
            np.log(
                np.clip(
                    probabilities[np.arange(len(y_true)), y_true], 1e-12, 1.0
                )
            )
        )
    )
    brier = float(np.mean(np.sum((probabilities - one_hot) ** 2, axis=1)))
    metrics = {
        "accuracy": float(accuracy_score(y_true, prediction)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, prediction)),
        "macro_precision": float(
            precision_score(y_true, prediction, average="macro", zero_division=0)
        ),
        "macro_recall": float(
            recall_score(y_true, prediction, average="macro", zero_division=0)
        ),
        "macro_f1": float(
            f1_score(y_true, prediction, average="macro", zero_division=0)
        ),
        "weighted_precision": float(
            precision_score(y_true, prediction, average="weighted", zero_division=0)
        ),
        "weighted_recall": float(
            recall_score(y_true, prediction, average="weighted", zero_division=0)
        ),
        "weighted_f1": float(
            f1_score(y_true, prediction, average="weighted", zero_division=0)
        ),
        "cohen_kappa": float(cohen_kappa_score(y_true, prediction)),
        "mcc": float(matthews_corrcoef(y_true, prediction)),
        "ece_15_bins": expected_calibration_error(y_true, probabilities, bins=15),
        "brier_multiclass": brier,
        "negative_log_likelihood": nll,
        "uncertainty_error_auroc": uncertainty_auc,
        "mean_uncertainty_correct": float(
            uncertainty[prediction == y_true].mean()
        ),
        "mean_uncertainty_incorrect": float(
            uncertainty[prediction != y_true].mean()
        )
        if np.any(prediction != y_true)
        else float("nan"),
        "inference_time_seconds": inference_seconds,
        "inference_time_ms_per_image": 1000.0
        * inference_seconds
        / len(y_true),
    }

    report = classification_report(
        y_true,
        prediction,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    pd.DataFrame(report).transpose().to_csv(
        output_dir / "classification_report.csv"
    )
    matrix = confusion_matrix(
        y_true, prediction, labels=np.arange(len(class_names))
    )
    pd.DataFrame(
        matrix,
        index=class_names,
        columns=class_names,
    ).to_csv(output_dir / "confusion_matrix.csv")
    save_evaluation_plots(
        y_true, prediction, probabilities, class_names, output_dir
    )

    prediction_table = pd.DataFrame(
        {
            "path": paths,
            "true_index": y_true,
            "true_class": [class_names[index] for index in y_true],
            "predicted_index": prediction,
            "predicted_class": [class_names[index] for index in prediction],
            "correct": prediction == y_true,
            "confidence": probabilities.max(axis=1),
            "uncertainty": uncertainty,
        }
    )
    for index, class_name in enumerate(class_names):
        prediction_table[f"probability_{class_name}"] = probabilities[:, index]
        if alpha is not None:
            prediction_table[f"evidence_{class_name}"] = alpha[:, index] - 1.0
    prediction_table.to_csv(output_dir / "holdout_predictions.csv", index=False)
    return metrics


def normalise_path(value: str | Path) -> str:
    return str(value).replace("\\", "/").rstrip("/").casefold()


def image_identifier_keys(value: str | Path) -> List[str]:
    """Return safe identifiers that remain stable after relocating a dataset.

    Absolute paths can change when the verified dataset is copied from C: to D:.
    The split/class/filename suffix remains stable and is therefore preferred.
    Filename and stem fallbacks are used only when they identify one fold.
    """
    normalised = normalise_path(str(value).strip())
    parts = [part for part in normalised.split("/") if part]
    if not parts:
        return []
    filename = parts[-1]
    stem = Path(filename).stem.casefold()
    keys = [normalised]
    for depth in (4, 3, 2):
        if len(parts) >= depth:
            keys.append("/".join(parts[-depth:]))
    keys.extend([filename, stem])
    return list(dict.fromkeys(keys))


def resolve_presplit_dataset_root(configured_path: str | Path) -> Path:
    """Locate a train/val/test root near the configured Windows path.

    This handles the common case where an extracted archive creates an extra
    directory level, or the notebook is moved beside the dataset.  It does not
    search entire drives or silently accept a folder without all three splits.
    """

    configured = Path(configured_path).expanduser()

    def is_presplit_root(candidate: Path) -> bool:
        return candidate.is_dir() and all(
            (candidate / split).is_dir() for split in ("train", "val", "test")
        )

    candidates: List[Path] = []
    seen = set()

    def add(candidate: Path) -> None:
        key = normalise_path(candidate)
        if key not in seen:
            seen.add(key)
            candidates.append(candidate)

    add(configured)
    add(Path.home() / "Downloads" / configured.name)
    add(Path.cwd() / configured.name)

    search_bases = [configured, configured.parent, Path.home() / "Downloads", Path.cwd()]
    for base_dir in search_bases:
        if not base_dir.is_dir():
            continue
        # Inspect at most two nearby directory levels. Prefer names that look
        # related to the configured dataset before considering other folders.
        children = sorted(
            (item for item in base_dir.iterdir() if item.is_dir()),
            key=lambda item: ("patient" not in item.name.casefold(), item.name.casefold()),
        )
        for child in children:
            add(child)
            try:
                grandchildren = [item for item in child.iterdir() if item.is_dir()]
            except OSError:
                grandchildren = []
            for grandchild in grandchildren:
                add(grandchild)

    for candidate in candidates:
        if is_presplit_root(candidate):
            resolved = candidate.resolve()
            if normalise_path(resolved) != normalise_path(configured):
                print(
                    "Configured dataset path was not usable. "
                    f"Automatically using detected split root: {resolved}"
                )
            return resolved

    checked = "\n".join(f"  - {candidate}" for candidate in candidates[:30])
    raise FileNotFoundError(
        "Could not locate the pre-split dataset. A valid root must contain "
        "train, val, and test subfolders.\n"
        f"Configured path: {configured}\n"
        "Checked nearby locations:\n"
        f"{checked}\n\n"
        "Set cfg.dataset to the actual parent folder that directly contains "
        "train, val, and test, then run the configuration and run cells again."
    )


def read_metadata_csv_flexible(csv_path: str | Path) -> pd.DataFrame:
    """Read comma-, semicolon-, or tab-delimited patient metadata."""
    csv_path = Path(csv_path)
    try:
        return pd.read_csv(csv_path, sep=None, engine="python")
    except Exception as error:
        raise ValueError(
            f"Unable to read patient metadata CSV: {csv_path}\n{error}"
        ) from error


def detect_metadata_columns(
    columns: Sequence[str], preferred_patient: str, preferred_image: str
) -> Tuple[str, str] | None:
    """Detect patient/image fields in either legacy or final-manifest schemas."""
    actual = {str(column).strip().casefold(): str(column) for column in columns}

    patient_candidates = (
        preferred_patient,
        "patient_id",
        "patient_num",
        "patient_number",
        "patient",
        "subject_id",
        "subject",
    )
    image_candidates = (
        preferred_image,
        "path",
        "image_path",
        "filepath",
        "file_path",
        "image_name",
        "filename",
        "file_name",
        "image",
    )

    patient_column = next(
        (actual[name.strip().casefold()] for name in patient_candidates
         if name.strip().casefold() in actual),
        None,
    )
    image_column = next(
        (actual[name.strip().casefold()] for name in image_candidates
         if name.strip().casefold() in actual),
        None,
    )
    if patient_column is None or image_column is None:
        return None
    return patient_column, image_column


def resolve_patient_metadata_csv(
    configured_path: str | Path,
    dataset_root: str | Path,
    source_results: str | Path,
    preferred_patient: str,
    preferred_image: str,
) -> Tuple[Path, str, str]:
    """Locate a usable patient CSV and determine its actual column names."""
    configured = Path(configured_path).expanduser()
    dataset_root = Path(dataset_root)
    source_results = Path(source_results).expanduser()

    candidates: List[Path] = []
    seen = set()

    def add(candidate: Path) -> None:
        key = normalise_path(candidate)
        if key not in seen:
            seen.add(key)
            candidates.append(candidate)

    for candidate in (
        configured,
        dataset_root / configured.name,
        dataset_root.parent / configured.name,
        Path.home() / "Downloads" / configured.name,
        Path.cwd() / configured.name,
        source_results / configured.name,
    ):
        add(candidate)

    search_directories = (
        dataset_root,
        dataset_root.parent,
        configured.parent,
        Path.home() / "Downloads",
        source_results,
        Path.cwd(),
    )
    for directory in search_directories:
        if not directory.is_dir():
            continue
        try:
            csv_files = list(directory.glob("*.csv"))
        except OSError:
            csv_files = []
        csv_files.sort(
            key=lambda path: (
                not any(word in path.name.casefold() for word in
                        ("patient", "split", "manifest", "metadata")),
                path.name.casefold(),
            )
        )
        for csv_file in csv_files:
            add(csv_file)

    readable_csvs = []
    for candidate in candidates:
        if not candidate.is_file():
            continue
        try:
            preview = pd.read_csv(candidate, sep=None, engine="python", nrows=5)
        except Exception:
            continue
        readable_csvs.append((candidate, list(preview.columns)))
        detected = detect_metadata_columns(
            preview.columns, preferred_patient, preferred_image
        )
        if detected is not None:
            patient_column, image_column = detected
            resolved = candidate.resolve()
            if normalise_path(resolved) != normalise_path(configured):
                print(
                    "Configured metadata CSV was not usable. "
                    f"Automatically using: {resolved}"
                )
            print(
                f"Patient metadata columns: patient='{patient_column}', "
                f"image='{image_column}'"
            )
            return resolved, patient_column, image_column

    discovered = "\n".join(
        f"  - {path}: {columns}" for path, columns in readable_csvs[:20]
    ) or "  - No readable CSV files were found in the nearby locations."
    raise FileNotFoundError(
        "Could not locate a patient metadata CSV with both an image/path column "
        "and a patient-ID column.\n"
        f"Configured CSV: {configured}\n"
        "Readable nearby CSV files and their columns:\n"
        f"{discovered}\n\n"
        "Set cfg.metadata_csv to the actual patient-split CSV. Supported schemas "
        "include Image_name/Patient_num and path/patient_id."
    )


def image_identifier_keys(value: str | Path) -> List[str]:
    """Generate full-path, suffix, filename, and stem keys for safe matching."""
    normalised = str(value).strip().replace("\\", "/").rstrip("/").casefold()
    parts = [part for part in normalised.split("/") if part]
    if not parts:
        return []
    filename = parts[-1]
    stem = Path(filename).stem.casefold()
    keys = [normalised]
    for depth in (4, 3, 2):
        if len(parts) >= depth:
            keys.append("/".join(parts[-depth:]))
    keys.extend([filename, stem])
    return list(dict.fromkeys(keys))


def load_patient_ids_flexible(
    raw_paths: np.ndarray,
    metadata_csv: str | Path,
    patient_column: str,
    image_column: str,
) -> Tuple[np.ndarray, pd.DataFrame]:
    """Map loaded images to patients across legacy and manifest CSV formats."""
    metadata = read_metadata_csv_flexible(metadata_csv)
    missing_columns = [
        column for column in (patient_column, image_column)
        if column not in metadata.columns
    ]
    if missing_columns:
        raise ValueError(
            f"Missing metadata columns: {missing_columns}. "
            f"Available columns: {list(metadata.columns)}"
        )
    if metadata[[patient_column, image_column]].isna().any().any():
        raise ValueError("Patient ID and image/path fields must not contain missing values.")

    metadata = metadata.copy()
    metadata[patient_column] = metadata[patient_column].astype(str).str.strip()
    key_to_patients: Dict[str, set] = {}
    for image_value, patient_value in zip(
        metadata[image_column].astype(str), metadata[patient_column]
    ):
        for key in image_identifier_keys(image_value):
            key_to_patients.setdefault(key, set()).add(patient_value)
    unique_lookup = {
        key: next(iter(patient_values))
        for key, patient_values in key_to_patients.items()
        if len(patient_values) == 1
    }

    patient_ids = []
    unmatched = []
    ambiguous = []
    for raw_path in raw_paths:
        keys = image_identifier_keys(raw_path)
        matches = [unique_lookup[key] for key in keys if key in unique_lookup]
        distinct = list(dict.fromkeys(matches))
        if len(distinct) == 1:
            patient_ids.append(distinct[0])
        elif len(distinct) > 1:
            ambiguous.append(str(raw_path))
        else:
            unmatched.append(str(raw_path))

    if ambiguous:
        raise ValueError(
            f"{len(ambiguous)} image paths matched multiple patient IDs. "
            f"Examples: {ambiguous[:5]}"
        )
    if unmatched:
        raise ValueError(
            f"{len(unmatched)} loaded images could not be matched to the patient CSV. "
            f"Examples: {unmatched[:5]}"
        )
    return np.asarray(patient_ids, dtype=object), metadata


def read_locked_folds(
    csv_path: Path,
    raw_paths: np.ndarray,
    development_idx: np.ndarray,
    outer_folds: int,
) -> np.ndarray:
    """Load locked folds even when the dataset root was relocated.

    Matching first considers stable path suffixes such as
    split/class/filename. Shorter fallbacks are retained only when they map
    unambiguously to one locked fold.
    """
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Locked fold file not found: {csv_path}\n"
            "Use the locked_patient_fold_assignments.csv from the completed QGECAF run."
        )
    table = pd.read_csv(csv_path)
    required = {"path", "outer_fold"}
    if not required.issubset(table.columns):
        raise ValueError(f"{csv_path} must contain columns {sorted(required)}")
    key_to_folds: Dict[str, set] = {}
    for path, fold in zip(table["path"], table["outer_fold"]):
        fold = int(fold)
        for key in image_identifier_keys(path):
            key_to_folds.setdefault(key, set()).add(fold)

    # Drop keys that refer to more than one fold; they are unsafe fallbacks.
    fold_map = {
        key: next(iter(folds))
        for key, folds in key_to_folds.items()
        if len(folds) == 1
    }
    assignments = np.full(len(raw_paths), -999, dtype=np.int32)
    missing = []
    ambiguous = []
    for index in development_idx:
        matches = [
            fold_map[key]
            for key in image_identifier_keys(raw_paths[index])
            if key in fold_map
        ]
        distinct = list(dict.fromkeys(matches))
        if len(distinct) == 1:
            assignments[index] = distinct[0]
        elif len(distinct) > 1:
            ambiguous.append(str(raw_paths[index]))
        else:
            missing.append(str(raw_paths[index]))
    if ambiguous:
        raise RuntimeError(
            f"{len(ambiguous)} development images matched multiple locked folds. "
            f"First ambiguous path: {ambiguous[0]}"
        )
    if missing:
        raise RuntimeError(
            f"{len(missing)} development images are absent from the locked fold file. "
            f"First missing path: {missing[0]}\n"
            "The locked-fold CSV must describe the same development images."
        )
    valid = set(assignments[development_idx].tolist())
    expected = set(range(1, outer_folds + 1))
    if valid != expected:
        raise RuntimeError(
            f"Locked folds are {sorted(valid)}, expected {sorted(expected)}."
        )
    return assignments


def build_base_config(cfg: RunConfig, base):
    return base.Config(
        dataset=cfg.dataset,
        metadata_csv=cfg.metadata_csv,
        patient_column=cfg.patient_column,
        image_column=cfg.image_column,
        output=cfg.output,
        image_size=cfg.image_size,
        seed=cfg.seed,
        pca_variance=cfg.pca_variance,
        bgwo_agents=cfg.bgwo_agents,
        bgwo_iters=cfg.bgwo_iters,
        bgwo_cv=cfg.bgwo_cv,
        bgwo_alpha=cfg.bgwo_alpha,
        embed_dim=cfg.embed_dim,
        attention_heads=cfg.attention_heads,
        batch_size=cfg.batch_size,
        epochs=cfg.epochs,
        learning_rate=cfg.learning_rate,
        kl_weight=cfg.kl_weight,
        dropout=cfg.dropout,
        fine_tune_last=cfg.fine_tune_last,
        force_features=cfg.force_features,
        smoke_test=False,
        outer_folds=cfg.outer_folds,
        inner_folds=cfg.inner_folds,
        early_stopping_patience=cfg.early_stopping_patience,
    )


SUMMARY_METRICS = [
    "accuracy",
    "balanced_accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_f1",
    "cohen_kappa",
    "mcc",
    "ece_15_bins",
    "brier_multiclass",
    "negative_log_likelihood",
    "uncertainty_error_auroc",
    "training_time_seconds",
    "mean_epoch_time_seconds",
    "inference_time_seconds",
    "inference_time_ms_per_image",
    "epochs_run",
]


def save_summaries(fold_metrics: pd.DataFrame, output_dir: Path) -> None:
    fold_metrics = fold_metrics.sort_values(["variant", "fold"])
    fold_metrics.to_csv(output_dir / "ablation_fold_metrics.csv", index=False)

    rows = []
    publication_rows = []
    for variant, group in fold_metrics.groupby("variant", sort=True):
        publication_row = {
            "variant": variant,
            "model_variant": VARIANT_INFO[variant],
        }
        for metric in SUMMARY_METRICS:
            values = pd.to_numeric(group[metric], errors="coerce").dropna().to_numpy(float)
            count = len(values)
            mean = float(np.mean(values)) if count else float("nan")
            sd = float(np.std(values, ddof=1)) if count > 1 else float("nan")
            half_width = (
                float(t.ppf(0.975, count - 1) * sd / math.sqrt(count))
                if count > 1
                else float("nan")
            )
            rows.append(
                {
                    "variant": variant,
                    "model_variant": VARIANT_INFO[variant],
                    "metric": metric,
                    "folds": count,
                    "mean": mean,
                    "sample_sd": sd,
                    "ci95_low": mean - half_width,
                    "ci95_high": mean + half_width,
                }
            )
            publication_row[metric] = f"{mean:.4f} ± {sd:.4f}"
        publication_rows.append(publication_row)

    pd.DataFrame(rows).to_csv(output_dir / "ablation_summary_long.csv", index=False)
    pd.DataFrame(publication_rows).to_csv(
        output_dir / "ablation_publication_table.csv", index=False
    )
    save_paired_statistics(fold_metrics, output_dir, metric="macro_f1")


def holm_adjust(p_values: Sequence[float]) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)
    running = 0.0
    total = len(p_values)
    for rank, original_index in enumerate(order):
        candidate = (total - rank) * p_values[original_index]
        running = max(running, candidate)
        adjusted[original_index] = min(1.0, running)
    return adjusted


def save_paired_statistics(
    fold_metrics: pd.DataFrame, output_dir: Path, metric: str
) -> None:
    pivot = fold_metrics.pivot(index="fold", columns="variant", values=metric)
    complete = pivot.dropna(axis=0, how="any")
    if complete.shape[0] < 2 or complete.shape[1] < 3:
        return
    statistic, p_value = friedmanchisquare(
        *[complete[column].to_numpy(float) for column in complete.columns]
    )
    omnibus = {
        "metric": metric,
        "folds": int(complete.shape[0]),
        "models": list(complete.columns),
        "friedman_statistic": float(statistic),
        "friedman_p": float(p_value),
    }
    (output_dir / f"friedman_{metric}.json").write_text(
        json.dumps(omnibus, indent=2), encoding="utf-8"
    )

    if "A9" not in complete.columns:
        return
    comparisons = []
    for variant in complete.columns:
        if variant == "A9":
            continue
        full = complete["A9"].to_numpy(float)
        ablated = complete[variant].to_numpy(float)
        try:
            result = wilcoxon(full, ablated, alternative="two-sided", zero_method="wilcox")
            raw_p = float(result.pvalue)
            test_statistic = float(result.statistic)
        except ValueError:
            raw_p = 1.0
            test_statistic = 0.0
        comparisons.append(
            {
                "reference": "A9",
                "comparison": variant,
                "mean_difference_A9_minus_ablation": float(np.mean(full - ablated)),
                "wilcoxon_statistic": test_statistic,
                "raw_p": raw_p,
            }
        )
    adjusted = holm_adjust([row["raw_p"] for row in comparisons])
    for row, adjusted_p in zip(comparisons, adjusted):
        row["holm_adjusted_p"] = float(adjusted_p)
        row["significant_at_0.05"] = bool(adjusted_p < 0.05)
    pd.DataFrame(comparisons).to_csv(
        output_dir / f"wilcoxon_holm_{metric}.csv", index=False
    )


## 4. Leakage-controlled patient-level cross-validation runner


In [4]:
def run(cfg: RunConfig) -> None:
    variants = [item.strip().upper() for item in cfg.variants.split(",") if item.strip()]
    unknown = sorted(set(variants).difference(VARIANT_INFO))
    if unknown:
        raise ValueError(f"Unknown variants: {unknown}. Valid IDs: {sorted(VARIANT_INFO)}")
    variants = list(dict.fromkeys(variants))

    base = load_publication_module(cfg.base_code)
    base.configure_gpu()
    set_seed(cfg.seed)

    output_dir = Path(cfg.output).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "ablation_configuration.json").write_text(
        json.dumps({**asdict(cfg), "variant_descriptions": VARIANT_INFO}, indent=2),
        encoding="utf-8",
    )
    base_cfg = build_base_config(cfg, base)

    dataset_root = resolve_presplit_dataset_root(cfg.dataset)
    print(f"Dataset root: {dataset_root}")
    paths, labels, class_names, split_names = base.discover_presplit_dataset(
        dataset_root
    )
    source_cache = Path(cfg.source_results) / "raw_features_presplit.npz"
    # Never overwrite the completed publication run when forced re-extraction
    # is requested; forced features are written inside the ablation directory.
    cache_path = (
        source_cache
        if source_cache.exists() and not cfg.force_features
        else output_dir / "raw_features_presplit.npz"
    )
    raw = base.extract_all_features(paths, labels, base_cfg, cache_path)
    labels = raw["labels"]
    train_idx, val_idx, test_idx = base.recover_presplit_indices(
        raw["paths"], paths, split_names
    )
    metadata_csv, patient_column, image_column = resolve_patient_metadata_csv(
        cfg.metadata_csv,
        dataset_root,
        cfg.source_results,
        cfg.patient_column,
        cfg.image_column,
    )
    patient_ids, _ = load_patient_ids_flexible(
        raw["paths"], metadata_csv, patient_column, image_column
    )
    base.validate_patient_separation(patient_ids, train_idx, val_idx, test_idx)
    development_idx = np.concatenate([train_idx, val_idx])
    if set(patient_ids[development_idx]).intersection(set(patient_ids[test_idx])):
        raise RuntimeError("Development/test patient leakage detected.")

    assignments = read_locked_folds(
        Path(cfg.source_results) / "locked_patient_fold_assignments.csv",
        raw["paths"],
        development_idx,
        cfg.outer_folds,
    )
    num_classes = len(class_names)
    all_rows = []

    for fold in range(1, cfg.outer_folds + 1):
        fold_root = output_dir / f"fold_{fold:02d}"
        pending = []
        for variant in variants:
            metrics_path = fold_root / variant / "fold_result.json"
            if cfg.resume and metrics_path.exists():
                all_rows.append(json.loads(metrics_path.read_text(encoding="utf-8")))
            else:
                pending.append(variant)
        if not pending:
            print(f"Fold {fold}: all requested variants already completed.")
            continue

        print("\n" + "=" * 78)
        print(f"ABLATION OUTER FOLD {fold}/{cfg.outer_folds}: {pending}")
        print("=" * 78)
        outer_holdout_idx = development_idx[assignments[development_idx] == fold]
        outer_train_idx = development_idx[assignments[development_idx] != fold]
        fold_cfg = replace(base_cfg, seed=cfg.seed + fold)
        # Use the original seed rule, random_state = 42 + fold.  Passing
        # fold_cfg here would incorrectly produce 42 + 2*fold.
        inner_train_idx, inner_val_idx = base.choose_inner_split(
            outer_train_idx, labels, patient_ids, base_cfg, fold
        )
        train_patients = set(patient_ids[inner_train_idx])
        val_patients = set(patient_ids[inner_val_idx])
        holdout_patients = set(patient_ids[outer_holdout_idx])
        if (
            train_patients & val_patients
            or train_patients & holdout_patients
            or val_patients & holdout_patients
        ):
            raise RuntimeError(f"Patient leakage detected in fold {fold}.")

        # Persist the exact image and patient membership used in this fold.
        split_audit = pd.concat(
            [
                pd.DataFrame({
                    "path": raw["paths"][inner_train_idx],
                    "patient_id": patient_ids[inner_train_idx],
                    "label": labels[inner_train_idx],
                    "role": "inner_train",
                }),
                pd.DataFrame({
                    "path": raw["paths"][inner_val_idx],
                    "patient_id": patient_ids[inner_val_idx],
                    "label": labels[inner_val_idx],
                    "role": "inner_validation",
                }),
                pd.DataFrame({
                    "path": raw["paths"][outer_holdout_idx],
                    "patient_id": patient_ids[outer_holdout_idx],
                    "label": labels[outer_holdout_idx],
                    "role": "outer_holdout",
                }),
            ],
            ignore_index=True,
        )
        split_audit.insert(0, "outer_fold", fold)
        split_audit.to_csv(fold_root / "patient_partition_audit.csv", index=False)

        preprocessing_dir = fold_root / "shared_preprocessing"
        preprocessing_dir.mkdir(parents=True, exist_ok=True)
        transformed_path = preprocessing_dir / "transformed_features.npz"
        processor_path = preprocessing_dir / "preprocessing.joblib"
        if cfg.resume and transformed_path.exists() and processor_path.exists():
            stored = np.load(transformed_path)
            transformed = {
                family: stored[family].astype(np.float32)
                for family in ("wst", "gabor", "lbp", "hog", "shape")
            }
            quality_z = stored["quality"].astype(np.float32)
        else:
            processors, transformed, quality_z = base.fit_preprocessing(
                raw, inner_train_idx, labels, patient_ids, fold_cfg
            )
            joblib.dump(processors, processor_path)
            np.savez_compressed(
                transformed_path, **transformed, quality=quality_z
            )

        all_train_inputs = base.make_inputs(
            raw, transformed, quality_z, inner_train_idx
        )
        all_val_inputs = base.make_inputs(
            raw, transformed, quality_z, inner_val_idx
        )
        all_holdout_inputs = base.make_inputs(
            raw, transformed, quality_z, outer_holdout_idx
        )
        y_train = tf.keras.utils.to_categorical(
            labels[inner_train_idx], num_classes
        )
        y_val = tf.keras.utils.to_categorical(labels[inner_val_idx], num_classes)
        feature_dims = {
            family: transformed[family].shape[1]
            for family in ("wst", "gabor", "lbp", "hog", "shape")
        }

        for variant in pending:
            variant_dir = fold_root / variant
            variant_dir.mkdir(parents=True, exist_ok=True)
            backup_dir = variant_dir / "training_backup"
            epoch_log_path = variant_dir / "epoch_log.csv"
            epoch_timing_path = variant_dir / "epoch_timing.csv"
            training_history_path = variant_dir / "training_history.csv"
            best_weights_path = variant_dir / "best.weights.h5"
            final_weights_path = variant_dir / "final_restored.weights.h5"
            training_complete_path = variant_dir / "training_complete.json"

            if not cfg.resume:
                # --no-resume deliberately starts this variant-fold from scratch.
                for stale_path in (
                    epoch_log_path,
                    epoch_timing_path,
                    training_history_path,
                    best_weights_path,
                    final_weights_path,
                    training_complete_path,
                    variant_dir / "fold_result.json",
                    variant_dir / "checkpoint_state.json",
                    variant_dir / "early_stopping_state.json",
                    variant_dir / "reduce_lr_state.json",
                ):
                    if stale_path.exists():
                        stale_path.unlink()
                if backup_dir.exists():
                    shutil.rmtree(backup_dir)

            print(f"\nFold {fold}, {variant}: {VARIANT_INFO[variant]}")
            tf.keras.backend.clear_session()
            set_seed(cfg.seed + fold)
            model = build_ablation_model(variant, feature_dims, num_classes, cfg)
            compile_ablation_model(
                model, variant, labels[inner_train_idx], num_classes, cfg, base
            )
            x_train = select_inputs(all_train_inputs, model)
            x_val = select_inputs(all_val_inputs, model)
            x_holdout = select_inputs(all_holdout_inputs, model)

            # If fitting had already ended but the process stopped before the
            # completion marker was written, reconstruct it from persistent logs.
            if (
                cfg.resume
                and not training_complete_path.exists()
                and best_weights_path.exists()
            ):
                prior_history = read_deduplicated_csv(epoch_log_path)
                prior_timing = read_deduplicated_csv(epoch_timing_path)
                early_state_path = variant_dir / "early_stopping_state.json"
                early_state = (
                    json.loads(early_state_path.read_text(encoding="utf-8"))
                    if early_state_path.exists()
                    else {}
                )
                stopped_early = int(early_state.get("stopped_epoch", 0)) > 0
                reached_maximum = (
                    not prior_history.empty
                    and int(prior_history["epoch"].max()) + 1 >= cfg.epochs
                )
                if (
                    (stopped_early or reached_maximum)
                    and not prior_timing.empty
                    and "val_loss" in prior_history.columns
                ):
                    print(
                        "Detected a completed fit without its completion marker; "
                        "recovering the global best checkpoint."
                    )
                    model.load_weights(best_weights_path)
                    model.save_weights(final_weights_path)
                    epochs_logged = int(prior_history["epoch"].max()) + 1
                    best_row = prior_history["val_loss"].astype(float).idxmin()
                    best_logged_epoch = int(prior_history.loc[best_row, "epoch"]) + 1
                    cumulative_seconds = float(
                        prior_timing["epoch_time_seconds"].astype(float).sum()
                    )
                    mean_seconds = float(
                        prior_timing["epoch_time_seconds"].astype(float).mean()
                    )
                    prior_history.to_csv(epoch_log_path, index=False)
                    prior_history.to_csv(training_history_path, index=False)
                    prior_timing.to_csv(epoch_timing_path, index=False)
                    write_json_atomic(
                        training_complete_path,
                        {
                            "fold": fold,
                            "variant": variant,
                            "epochs_run": epochs_logged,
                            "best_epoch": best_logged_epoch,
                            "training_time_seconds": cumulative_seconds,
                            "mean_epoch_time_seconds": mean_seconds,
                            "timing_definition": (
                                "Sum of wall-clock durations of all completed epochs, "
                                "including training and validation; feature preprocessing excluded."
                            ),
                            "reconstructed_after_interruption": True,
                        },
                    )

            resumed_after_completed_training = (
                cfg.resume
                and training_complete_path.exists()
                and final_weights_path.exists()
            )
            if resumed_after_completed_training:
                print(
                    "Training was already completed; loading restored best weights "
                    "and continuing with holdout evaluation."
                )
                model.load_weights(final_weights_path)
                training_state = json.loads(
                    training_complete_path.read_text(encoding="utf-8")
                )
                epoch_frame = read_deduplicated_csv(epoch_timing_path)
                history_frame = read_deduplicated_csv(epoch_log_path)
                training_seconds = float(training_state["training_time_seconds"])
                epochs_run = int(training_state["epochs_run"])
                best_epoch = int(training_state["best_epoch"])
                mean_epoch_seconds = float(training_state["mean_epoch_time_seconds"])
            else:
                epoch_timer = PersistentEpochTimer(epoch_timing_path)
                checkpoint = PersistentModelCheckpoint(
                    str(best_weights_path),
                    state_path=variant_dir / "checkpoint_state.json",
                    monitor="val_loss",
                    save_best_only=True,
                    save_weights_only=True,
                    verbose=1,
                )
                early_stopping = PersistentEarlyStopping(
                    state_path=variant_dir / "early_stopping_state.json",
                    monitor="val_loss",
                    patience=cfg.early_stopping_patience,
                    restore_best_weights=False,
                    verbose=1,
                )
                reduce_lr = PersistentReduceLROnPlateau(
                    state_path=variant_dir / "reduce_lr_state.json",
                    monitor="val_loss",
                    factor=0.5,
                    patience=cfg.reduce_lr_patience,
                    min_lr=1e-7,
                    verbose=1,
                )
                callbacks = [
                    epoch_timer,
                    tf.keras.callbacks.CSVLogger(
                        str(epoch_log_path), append=cfg.resume
                    ),
                    checkpoint,
                    early_stopping,
                    reduce_lr,
                    tf.keras.callbacks.BackupAndRestore(
                        backup_dir=str(backup_dir),
                        save_freq="epoch",
                        delete_checkpoint=True,
                    ),
                ]
                model.fit(
                    x_train,
                    y_train,
                    validation_data=(x_val, y_val),
                    epochs=cfg.epochs,
                    batch_size=cfg.batch_size,
                    callbacks=callbacks,
                    verbose=1,
                )

                # Always evaluate the global best validation-loss checkpoint,
                # including when training crossed one or more process restarts.
                if not best_weights_path.exists():
                    raise RuntimeError(
                        f"Best checkpoint was not created for fold {fold}, {variant}."
                    )
                model.load_weights(best_weights_path)
                model.save_weights(final_weights_path)

                epoch_frame = read_deduplicated_csv(epoch_timing_path)
                history_frame = read_deduplicated_csv(epoch_log_path)
                if epoch_frame.empty:
                    raise RuntimeError("No epoch timing records were saved.")
                if history_frame.empty or "val_loss" not in history_frame.columns:
                    raise RuntimeError("No validation-loss history was saved.")

                # Rewrite de-duplicated logs after a successful resumed fit.
                epoch_frame.to_csv(epoch_timing_path, index=False)
                history_frame.to_csv(epoch_log_path, index=False)
                history_frame.to_csv(training_history_path, index=False)

                epochs_run = int(history_frame["epoch"].max()) + 1
                best_history_row = history_frame["val_loss"].astype(float).idxmin()
                best_epoch = int(history_frame.loc[best_history_row, "epoch"]) + 1
                training_seconds = float(
                    epoch_frame["epoch_time_seconds"].astype(float).sum()
                )
                mean_epoch_seconds = float(
                    epoch_frame["epoch_time_seconds"].astype(float).mean()
                )
                write_json_atomic(
                    training_complete_path,
                    {
                        "fold": fold,
                        "variant": variant,
                        "epochs_run": epochs_run,
                        "best_epoch": best_epoch,
                        "training_time_seconds": training_seconds,
                        "mean_epoch_time_seconds": mean_epoch_seconds,
                        "timing_definition": (
                            "Sum of wall-clock durations of all completed epochs, "
                            "including training and validation; feature preprocessing excluded."
                        ),
                    },
                )

            save_training_curves(history_frame, variant_dir)

            metrics = timed_evaluation(
                model,
                variant,
                x_holdout,
                labels[outer_holdout_idx],
                class_names,
                raw["paths"][outer_holdout_idx],
                variant_dir,
                cfg,
            )
            row = {
                "variant": variant,
                "model_variant": VARIANT_INFO[variant],
                "fold": fold,
                "seed": cfg.seed + fold,
                "best_epoch": best_epoch,
                "epochs_run": epochs_run,
                "training_time_seconds": training_seconds,
                "mean_epoch_time_seconds": mean_epoch_seconds,
                "training_images": len(inner_train_idx),
                "validation_images": len(inner_val_idx),
                "outer_holdout_images": len(outer_holdout_idx),
                "training_patients": len(train_patients),
                "validation_patients": len(val_patients),
                "outer_holdout_patients": len(holdout_patients),
                "trainable_parameters": int(
                    np.sum([np.prod(variable.shape) for variable in model.trainable_weights])
                ),
                "total_parameters": int(model.count_params()),
                **metrics,
            }
            write_json_atomic(variant_dir / "fold_result.json", row)
            write_json_atomic(
                variant_dir / "FOLD_COMPLETE.json",
                {
                    "completed": True,
                    "variant": variant,
                    "fold": fold,
                    "outer_holdout_images": len(outer_holdout_idx),
                    "outer_holdout_patients": len(holdout_patients),
                    "untouched_final_test_evaluated": False,
                },
            )
            all_rows.append(row)
            save_summaries(pd.DataFrame(all_rows), output_dir)

            del model, x_train, x_val, x_holdout
            gc.collect()
            tf.keras.backend.clear_session()

        del transformed, quality_z, all_train_inputs, all_val_inputs, all_holdout_inputs
        gc.collect()

    results = pd.DataFrame(all_rows).drop_duplicates(
        subset=["variant", "fold"], keep="last"
    )
    save_summaries(results, output_dir)
    complete_counts = results.groupby("variant")["fold"].nunique().to_dict()
    (output_dir / "RUN_STATUS.json").write_text(
        json.dumps(
            {
                "requested_variants": variants,
                "completed_folds_by_variant": complete_counts,
                "expected_folds": cfg.outer_folds,
                "untouched_test_evaluated": False,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    print("\nAblation study fold counts:")
    print(json.dumps(complete_counts, indent=2))
    print(f"\nResults saved to: {output_dir}")


## 5. Configure and verify the working QGECAF experiment

This cell points to the same clean dataset and the same locked folds created by the working QGECAF notebook. It refuses to continue unless the expected 10,203/1,062/1,135 physical split counts, 11,265 development assignments, 1,135 untouched-test assignments, ten outer fold IDs, and one-fold-per-patient constraint are all satisfied.


In [5]:
# ---------------------------------------------------------------------
# USER PATHS — same clean dataset and same completed/active QGECAF run
# ---------------------------------------------------------------------
from pathlib import Path
import os

PROJECT_ROOT = Path(r"D:\chaitali\qgacef")
DATASET_ROOT = Path(r"D:\chaitali\patientsplit801010_clean_v2")
METADATA_CSV = DATASET_ROOT / "patient_split_80_10_10.csv"

# This directory is produced by the attached working QGECAF notebook.
SOURCE_RESULTS = PROJECT_ROOT / "QGECAF_FROM_SCRATCH_RUN_01"
LOCKED_FOLD_FILE = SOURCE_RESULTS / "locked_patient_fold_assignments.csv"

# Keep ablation outputs separate from the proposed-model outputs.
ABLATION_OUTPUT = PROJECT_ROOT / "QGECAF_ABLATION_SAME_LOCKED_PATIENTS"

# The notebook may have been downloaded with or without a numeric suffix.
BASE_NOTEBOOK_CANDIDATES = [
    PROJECT_ROOT / "resumeQGECAF_CleanV2_FromScratch_Resumable_FIXED.ipynb",
    PROJECT_ROOT / "resumeQGECAF_CleanV2_FromScratch_Resumable_FIXED(3).ipynb",
    PROJECT_ROOT / "resumeQGECAF_CleanV2_FromScratch_Resumable_FIXED(2).ipynb",
    PROJECT_ROOT / "resumeQGECAF_CleanV2_FromScratch_Resumable_FIXED(1).ipynb",
]
BASE_NOTEBOOK = next(
    (path for path in BASE_NOTEBOOK_CANDIDATES if path.is_file()),
    None,
)

if BASE_NOTEBOOK is None:
    checked = "\n".join(f"  - {path}" for path in BASE_NOTEBOOK_CANDIDATES)
    raise FileNotFoundError(
        "The working QGECAF notebook was not found. Put it in PROJECT_ROOT.\n"
        f"Checked:\n{checked}"
    )

cfg = RunConfig(
    base_code=str(BASE_NOTEBOOK),
    dataset=str(DATASET_ROOT),
    metadata_csv=str(METADATA_CSV),
    source_results=str(SOURCE_RESULTS),
    output=str(ABLATION_OUTPUT),
    patient_column="Patient_num",
    image_column="Image_name",
    variants="A0,A1,A2,A3,A4,A5,A6,A7,A8,A9",
    image_size=128,
    seed=42,
    pca_variance=0.98,
    bgwo_agents=12,
    bgwo_iters=25,
    bgwo_cv=3,
    bgwo_alpha=0.99,
    embed_dim=64,
    attention_heads=4,
    attention_dropout=0.10,
    batch_size=8,
    inference_batch_size=16,
    epochs=50,
    learning_rate=1e-4,
    kl_weight=0.001,
    dropout=0.30,
    fine_tune_last=30,
    outer_folds=10,
    inner_folds=9,
    early_stopping_patience=8,
    reduce_lr_patience=3,
    force_features=False,
    resume=True,
)

# ---------------------------------------------------------------------
# Preflight 1: required paths and physical image counts
# ---------------------------------------------------------------------
if not METADATA_CSV.is_file():
    raise FileNotFoundError(f"Patient metadata CSV was not found: {METADATA_CSV}")
if not LOCKED_FOLD_FILE.is_file():
    raise FileNotFoundError(
        f"Locked patient folds were not found: {LOCKED_FOLD_FILE}\n"
        "Run the working QGECAF notebook at least through fold creation first."
    )

validated_dataset_root = resolve_presplit_dataset_root(cfg.dataset)
expected_counts = {"train": 10203, "val": 1062, "test": 1135}
image_extensions = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
actual_counts = {}
for split_name in ("train", "val", "test"):
    split_dir = validated_dataset_root / split_name
    actual_counts[split_name] = sum(
        1
        for path in split_dir.rglob("*")
        if path.is_file() and path.suffix.casefold() in image_extensions
    )
print("Dataset root:", validated_dataset_root)
print("Physical image counts:", actual_counts)
if actual_counts != expected_counts:
    raise RuntimeError(
        f"Unexpected split counts. Expected {expected_counts}, found {actual_counts}."
    )

# ---------------------------------------------------------------------
# Preflight 2: locked-fold integrity and patient leakage audit
# ---------------------------------------------------------------------
locked = pd.read_csv(LOCKED_FOLD_FILE)
required_locked_columns = {
    "path", "patient_id", "label", "class_name", "original_split", "outer_fold"
}
missing_locked_columns = required_locked_columns.difference(locked.columns)
if missing_locked_columns:
    raise ValueError(
        f"Locked fold file lacks columns: {sorted(missing_locked_columns)}"
    )
locked["outer_fold"] = pd.to_numeric(
    locked["outer_fold"], errors="raise"
).astype(int)
locked["patient_id"] = locked["patient_id"].astype(str)

development_locked = locked[locked["outer_fold"].between(1, 10)].copy()
final_test_locked = locked[locked["outer_fold"] == -1].copy()
if len(development_locked) != 11265:
    raise RuntimeError(
        f"Expected 11,265 locked development images; found {len(development_locked)}."
    )
if len(final_test_locked) != 1135:
    raise RuntimeError(
        f"Expected 1,135 untouched-test images; found {len(final_test_locked)}."
    )
if development_locked["patient_id"].nunique() != 1612:
    raise RuntimeError("Expected 1,612 development patients.")
if final_test_locked["patient_id"].nunique() != 180:
    raise RuntimeError("Expected 180 untouched-test patients.")
if set(development_locked["patient_id"]) & set(final_test_locked["patient_id"]):
    raise RuntimeError("Development/final-test patient leakage detected.")
if set(development_locked["outer_fold"].unique()) != set(range(1, 11)):
    raise RuntimeError("Locked development assignments must contain folds 1 through 10.")
patient_fold_counts = locked.groupby("patient_id")["outer_fold"].nunique()
if int(patient_fold_counts.max()) != 1:
    leaking = patient_fold_counts[patient_fold_counts > 1].index.tolist()[:10]
    raise RuntimeError(f"Patients assigned to multiple outer folds: {leaking}")

fold_counts = (
    development_locked.groupby("outer_fold")
    .agg(images=("path", "size"), patients=("patient_id", "nunique"))
    .reset_index()
)
print("\nLocked outer-fold counts:")
display(fold_counts)
print("PASS: same 11,265 development images and same patient folds are locked.")
print("PASS: untouched 1,135-image test cohort is excluded from ablation CV.")

# ---------------------------------------------------------------------
# Preflight 3: metadata schema and optional reusable feature cache
# ---------------------------------------------------------------------
detected_metadata_csv, detected_patient_column, detected_image_column = (
    resolve_patient_metadata_csv(
        cfg.metadata_csv,
        validated_dataset_root,
        cfg.source_results,
        cfg.patient_column,
        cfg.image_column,
    )
)
print("Metadata CSV:", detected_metadata_csv)
print("Patient column:", detected_patient_column)
print("Image column:", detected_image_column)

source_cache = SOURCE_RESULTS / "raw_features_presplit.npz"
if source_cache.exists():
    print("Reusable deterministic raw-feature cache:", source_cache)
else:
    print("No source feature cache found; features will be extracted into the ablation output.")

print("\nConfiguration:")
display(pd.Series(asdict(cfg), name="Value"))


Dataset root: D:\chaitali\patientsplit801010_clean_v2
Physical image counts: {'train': 10203, 'val': 1062, 'test': 1135}

Locked outer-fold counts:


,outer_fold,images,patients
0,1,1011,158
1,2,1123,164
2,3,1252,168
3,4,941,163
4,5,1139,166
5,6,1076,157
6,7,1279,161
7,8,1100,157
8,9,1184,161
9,10,1160,157


PASS: same 11,265 development images and same patient folds are locked.
PASS: untouched 1,135-image test cohort is excluded from ablation CV.
Patient metadata columns: patient='Patient_num', image='Image_name'
Metadata CSV: D:\chaitali\patientsplit801010_clean_v2\patient_split_80_10_10.csv
Patient column: Patient_num
Image column: Image_name
Reusable deterministic raw-feature cache: D:\chaitali\qgacef\QGECAF_FROM_SCRATCH_RUN_01\raw_features_presplit.npz

Configuration:


base_code                  D:\chaitali\qgacef\resumeQGECAF_CleanV2_FromSc...
dataset                              D:\chaitali\patientsplit801010_clean_v2
metadata_csv               D:\chaitali\patientsplit801010_clean_v2\patien...
source_results                 D:\chaitali\qgacef\QGECAF_FROM_SCRATCH_RUN_01
output                     D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED...
patient_column                                                   Patient_num
image_column                                                      Image_name
variants                                       A0,A1,A2,A3,A4,A5,A6,A7,A8,A9
image_size                                                               128
seed                                                                      42
pca_variance                                                            0.98
bgwo_agents                                                               12
bgwo_iters                                                                25

In [6]:
from pathlib import Path

ablation_root = Path(
    r"D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS"
)

ablation_root.mkdir(parents=True, exist_ok=True)

for fold in range(1, 11):
    fold_directory = ablation_root / f"fold_{fold:02d}"
    fold_directory.mkdir(parents=True, exist_ok=True)
    print("Created/verified:", fold_directory)

print("\nPASS: All ten fold directories are ready.")

Created/verified: D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS\fold_01
Created/verified: D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS\fold_02
Created/verified: D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS\fold_03
Created/verified: D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS\fold_04
Created/verified: D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS\fold_05
Created/verified: D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS\fold_06
Created/verified: D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS\fold_07
Created/verified: D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS\fold_08
Created/verified: D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS\fold_09
Created/verified: D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS\fold_10

PASS: All ten fold directories are ready.


In [7]:
import sys
import tensorflow as tf

print("Python executable:", sys.executable)
print("TensorFlow version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

Python executable: C:\Users\viraj\anaconda3\envs\chaitalienv\python.exe
TensorFlow version: 2.10.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 6. Run or safely resume the complete ablation study

This performs 100 paired training runs (10 variants × 10 locked outer folds). The shared fold-specific preprocessing is fitted only on each inner-training partition. If the kernel or computer stops, reopen the notebook, run the earlier cells, and run this cell again with `resume=True`.


In [8]:
# Run all A0–A9 variants on the same ten locked patient folds.
# The untouched final test cohort is never passed to model.fit or timed_evaluation.
run(cfg)



TensorFlow 2.10.0; GPUs detected: 1
Dataset root: D:\chaitali\patientsplit801010_clean_v2
Fixed patient-level split detected

TRAIN
  Fetal abdomen: 587 images
  Fetal brain: 2505 images
  Fetal femur: 861 images
  Fetal thorax: 1409 images
  Maternal cervix: 1293 images
  Other: 3548 images

VALIDATION
  Fetal abdomen: 62 images
  Fetal brain: 272 images
  Fetal femur: 97 images
  Fetal thorax: 154 images
  Maternal cervix: 165 images
  Other: 312 images

TEST
  Fetal abdomen: 62 images
  Fetal brain: 315 images
  Fetal femur: 82 images
  Fetal thorax: 155 images
  Maternal cervix: 168 images
  Other: 353 images
Loading cached features: D:\chaitali\qgacef\QGECAF_FROM_SCRATCH_RUN_01\raw_features_presplit.npz
Patient metadata columns: patient='Patient_num', image='Image_name'
Patient leakage audit passed: train, validation and test are disjoint.
Fold 1: all requested variants already completed.
Fold 2: all requested variants already completed.
Fold 3: all requested variants already comp

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



1131/1132 [============================>.] - ETA: 0s - loss: 0.0592 - accuracy: 0.9894
Epoch 12: val_loss did not improve from 0.25453

Epoch 12: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-05.
1132/1132 [==============================] - 28s 25ms/step - loss: 0.0596 - accuracy: 0.9893 - val_loss: 0.2983 - val_accuracy: 0.8950 - lr: 5.0000e-05
Epoch 13/50
1131/1132 [============================>.] - ETA: 0s - loss: 0.0520 - accuracy: 0.9916
Epoch 13: val_loss did not improve from 0.25453
1132/1132 [==============================] - 28s 25ms/step - loss: 0.0520 - accuracy: 0.9916 - val_loss: 0.2815 - val_accuracy: 0.9015 - lr: 2.5000e-05
Epoch 14/50
1130/1132 [============================>.] - ETA: 0s - loss: 0.0491 - accuracy: 0.9933
Epoch 14: val_loss did not improve from 0.25453
1132/1132 [==============================] - 29s 25ms/step - loss: 0.0491 - accuracy: 0.9933 - val_loss: 0.2835 - val_accuracy: 0.9136 - lr: 2.5000e-05
Epoch 14: early stopping

Fold 5, A5:

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



1131/1132 [============================>.] - ETA: 0s - loss: 0.1452 - accuracy: 0.9628
Epoch 5: val_loss did not improve from 0.27153
1132/1132 [==============================] - 32s 28ms/step - loss: 0.1451 - accuracy: 0.9628 - val_loss: 0.3097 - val_accuracy: 0.8727 - lr: 1.0000e-04
Epoch 6/50
1132/1132 [==============================] - ETA: 0s - loss: 0.1244 - accuracy: 0.9696
Epoch 6: val_loss improved from 0.27153 to 0.26445, saving model to D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS\fold_05\A9\best.weights.h5
1132/1132 [==============================] - 32s 29ms/step - loss: 0.1244 - accuracy: 0.9696 - val_loss: 0.2645 - val_accuracy: 0.9043 - lr: 1.0000e-04
Epoch 7/50
1132/1132 [==============================] - ETA: 0s - loss: 0.1107 - accuracy: 0.9699
Epoch 7: val_loss improved from 0.26445 to 0.22987, saving model to D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS\fold_05\A9\best.weights.h5
1132/1132 [==============================] - 32s 28ms/step - los

...

1127/1127 [==============================] - ETA: 0s - loss: 0.3507 - accuracy: 0.9042
Epoch 2: val_loss improved from 0.72608 to 0.32576, saving model to D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS\fold_09\A9\best.weights.h5
1127/1127 [==============================] - 33s 29ms/step - loss: 0.3507 - accuracy: 0.9042 - val_loss: 0.3258 - val_accuracy: 0.8783 - lr: 1.0000e-04
Epoch 3/50
1126/1127 [============================>.] - ETA: 0s - loss: 0.2566 - accuracy: 0.9302
Epoch 3: val_loss improved from 0.32576 to 0.25669, saving model to D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS\fold_09\A9\best.weights.h5
1127/1127 [==============================] - 33s 29ms/step - loss: 0.2566 - accuracy: 0.9302 - val_loss: 0.2567 - val_accuracy: 0.9073 - lr: 1.0000e-04
Epoch 4/50
 411/1127 [=========>....................] - ETA: 19s - loss: 0.2073 - accuracy: 0.9440

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



1127/1127 [==============================] - ETA: 0s - loss: 0.0398 - accuracy: 0.9952 
Epoch 18: val_loss did not improve from 0.17435
1127/1127 [==============================] - 11008s 10s/step - loss: 0.0398 - accuracy: 0.9952 - val_loss: 0.1885 - val_accuracy: 0.9316 - lr: 2.5000e-05
Epoch 19/50
1126/1127 [============================>.] - ETA: 0s - loss: 0.0379 - accuracy: 0.9950
Epoch 19: val_loss did not improve from 0.17435
1127/1127 [==============================] - 37s 33ms/step - loss: 0.0379 - accuracy: 0.9950 - val_loss: 0.1972 - val_accuracy: 0.9307 - lr: 2.5000e-05
Epoch 20/50
1126/1127 [============================>.] - ETA: 0s - loss: 0.0379 - accuracy: 0.9950
Epoch 20: val_loss did not improve from 0.17435

Epoch 20: ReduceLROnPlateau reducing learning rate to 1.249999968422344e-05.
1127/1127 [==============================] - 37s 32ms/step - loss: 0.0379 - accuracy: 0.9950 - val_loss: 0.2111 - val_accuracy: 0.9316 - lr: 2.5000e-05
Epoch 21/50
1126/1127 [===========

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



1143/1144 [============================>.] - ETA: 0s - loss: 0.0613 - accuracy: 0.9897
Epoch 12: val_loss improved from 0.17854 to 0.17625, saving model to D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS\fold_10\A6\best.weights.h5
1144/1144 [==============================] - 41s 36ms/step - loss: 0.0613 - accuracy: 0.9897 - val_loss: 0.1762 - val_accuracy: 0.9374 - lr: 5.0000e-05
Epoch 13/50
1143/1144 [============================>.] - ETA: 0s - loss: 0.0580 - accuracy: 0.9897
Epoch 13: val_loss did not improve from 0.17625
1144/1144 [==============================] - 40s 35ms/step - loss: 0.0581 - accuracy: 0.9897 - val_loss: 0.1823 - val_accuracy: 0.9406 - lr: 5.0000e-05
Epoch 14/50
1143/1144 [============================>.] - ETA: 0s - loss: 0.0593 - accuracy: 0.9895
Epoch 14: val_loss did not improve from 0.17625
1144/1144 [==============================] - 41s 36ms/step - loss: 0.0593 - accuracy: 0.9895 - val_loss: 0.1863 - val_accuracy: 0.9406 - lr: 5.0000e-05
Epoch 15/50

## 7. Display fold-wise, aggregate, and paired statistical results


In [9]:
result_root = Path(cfg.output)
fold_file = result_root / "ablation_fold_metrics.csv"
publication_file = result_root / "ablation_publication_table.csv"
long_summary_file = result_root / "ablation_summary_long.csv"
friedman_file = result_root / "friedman_macro_f1.json"
posthoc_file = result_root / "wilcoxon_holm_macro_f1.csv"

if fold_file.exists():
    print("Fold-wise metrics")
    display(pd.read_csv(fold_file))
else:
    print("Fold metrics are not available yet:", fold_file)

if publication_file.exists():
    print("\nMean ± sample SD across the ten paired folds")
    display(pd.read_csv(publication_file))

if long_summary_file.exists():
    print("\nMean, sample SD and 95% confidence intervals")
    display(pd.read_csv(long_summary_file))

if friedman_file.exists():
    print("\nFriedman test for macro F1")
    display(json.loads(friedman_file.read_text(encoding="utf-8")))

if posthoc_file.exists():
    print("\nPaired Wilcoxon tests versus A9 with Holm correction")
    display(pd.read_csv(posthoc_file))


Fold-wise metrics


,variant,model_variant,fold,seed,best_epoch,epochs_run,training_time_seconds,mean_epoch_time_seconds,training_images,validation_images,...,cohen_kappa,mcc,ece_15_bins,brier_multiclass,negative_log_likelihood,uncertainty_error_auroc,mean_uncertainty_correct,mean_uncertainty_incorrect,inference_time_seconds,inference_time_ms_per_image
0,A0,MobileNetV2 deep stream only,1,43,8,16,382.689232,23.918077,9142,1112,...,0.915363,0.916562,0.055516,0.111633,0.274572,0.723785,0.093252,0.106644,2.268782,2.244097
1,A0,MobileNetV2 deep stream only,2,44,8,16,376.927051,23.557941,9034,1108,...,0.903516,0.904065,0.051539,0.117541,0.286618,0.758920,0.090967,0.105861,2.280853,2.031036
2,A0,MobileNetV2 deep stream only,3,45,21,29,610.384156,21.047730,8903,1110,...,0.917354,0.917567,0.014046,0.108836,0.269066,0.803877,0.047797,0.057825,2.200373,1.757486
3,A0,MobileNetV2 deep stream only,4,46,15,23,647.882604,28.168809,9239,1085,...,0.897846,0.898962,0.016710,0.129779,0.314209,0.738302,0.050434,0.059845,3.055490,3.247067
4,A0,MobileNetV2 deep stream only,5,47,22,30,611.256646,20.375222,9050,1076,...,0.891367,0.891639,0.015265,0.143189,0.343001,0.803802,0.054860,0.065023,2.151390,1.888841
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,A9,Full QGECAF: quality gate and evidential loss ...,6,48,15,23,1294.124683,56.266291,9075,1114,...,0.902814,0.903977,0.023398,0.125821,0.305270,0.700949,0.053631,0.061076,2.963568,2.754245
96,A9,Full QGECAF: quality gate and evidential loss ...,7,49,12,20,637.815690,31.890784,8832,1154,...,0.909398,0.909987,0.027649,0.107200,0.261731,0.702904,0.063689,0.072586,2.681417,2.096495
97,A9,Full QGECAF: quality gate and evidential loss ...,8,50,7,15,497.779883,33.185326,9172,993,...,0.880680,0.883326,0.048941,0.142004,0.337517,0.735045,0.098239,0.114565,2.432948,2.211771
98,A9,Full QGECAF: quality gate and evidential loss ...,9,51,14,22,11704.305850,532.013902,9013,1068,...,0.927530,0.927869,0.026988,0.099125,0.248182,0.811068,0.056305,0.069381,1.262064,1.065933



Mean ± sample SD across the ten paired folds


,variant,model_variant,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,cohen_kappa,mcc,ece_15_bins,brier_multiclass,negative_log_likelihood,uncertainty_error_auroc,training_time_seconds,mean_epoch_time_seconds,inference_time_seconds,inference_time_ms_per_image,epochs_run
0,A0,MobileNetV2 deep stream only,0.9284 ± 0.0102,0.9359 ± 0.0138,0.9018 ± 0.0134,0.9359 ± 0.0138,0.9165 ± 0.0115,0.9287 ± 0.0103,0.9083 ± 0.0118,0.9091 ± 0.0117,0.0271 ± 0.0153,0.1205 ± 0.0147,0.3012 ± 0.0337,0.7557 ± 0.0435,527.6210 ± 127.6973,22.2918 ± 2.5010,2.2202 ± 0.5735,1.9952 ± 0.5930,23.8000 ± 5.7116
1,A1,Handcrafted stream only,0.8628 ± 0.0097,0.8664 ± 0.0171,0.8181 ± 0.0126,0.8664 ± 0.0171,0.8369 ± 0.0127,0.8645 ± 0.0097,0.8256 ± 0.0112,0.8280 ± 0.0108,0.0663 ± 0.0206,0.2077 ± 0.0171,0.4635 ± 0.0383,0.8166 ± 0.0218,372.6064 ± 58.2773,16.7629 ± 1.9226,0.9096 ± 0.1666,0.8163 ± 0.1798,22.3000 ± 2.9458
2,A2,Deep and handcrafted concatenation,0.9199 ± 0.0170,0.9318 ± 0.0135,0.8902 ± 0.0203,0.9318 ± 0.0135,0.9066 ± 0.0183,0.9202 ± 0.0176,0.8978 ± 0.0199,0.8995 ± 0.0184,0.0358 ± 0.0154,0.1264 ± 0.0260,0.3057 ± 0.0540,0.7285 ± 0.0420,3256.7469 ± 5493.9763,175.0821 ± 301.5575,2.8362 ± 0.8282,2.5458 ± 0.7962,21.1000 ± 7.0624
3,A3,Deep-query cross-attention only,0.9259 ± 0.0141,0.9347 ± 0.0096,0.8987 ± 0.0214,0.9347 ± 0.0096,0.9135 ± 0.0155,0.9267 ± 0.0133,0.9052 ± 0.0176,0.9062 ± 0.0166,0.0322 ± 0.0223,0.1215 ± 0.0218,0.3003 ± 0.0488,0.7671 ± 0.0328,936.6106 ± 552.7339,47.0895 ± 31.7420,2.8187 ± 0.6032,2.5470 ± 0.7076,21.3000 ± 4.7152
4,A4,Handcrafted-query cross-attention only,0.9238 ± 0.0151,0.9327 ± 0.0118,0.8952 ± 0.0213,0.9327 ± 0.0118,0.9106 ± 0.0164,0.9244 ± 0.0150,0.9026 ± 0.0182,0.9038 ± 0.0172,0.0417 ± 0.0238,0.1235 ± 0.0224,0.3030 ± 0.0449,0.6944 ± 0.0550,605.0253 ± 141.1053,33.9190 ± 4.2217,2.8314 ± 0.8494,2.5580 ± 0.9017,17.8000 ± 3.1903
5,A5,Bidirectional attention with fixed averaging,0.9261 ± 0.0118,0.9333 ± 0.0142,0.8976 ± 0.0130,0.9333 ± 0.0142,0.9131 ± 0.0125,0.9267 ± 0.0119,0.9053 ± 0.0148,0.9060 ± 0.0147,0.0352 ± 0.0214,0.1196 ± 0.0174,0.2957 ± 0.0359,0.7389 ± 0.0508,726.4958 ± 147.4680,35.9166 ± 4.4175,3.0183 ± 0.9286,2.7447 ± 1.0533,20.5000 ± 4.9497
6,A6,Bidirectional attention with learned global gate,0.9254 ± 0.0137,0.9333 ± 0.0111,0.8986 ± 0.0199,0.9333 ± 0.0111,0.9132 ± 0.0138,0.9257 ± 0.0138,0.9046 ± 0.0161,0.9057 ± 0.0155,0.0397 ± 0.0176,0.1225 ± 0.0231,0.3015 ± 0.0531,0.7554 ± 0.0527,1181.7284 ± 1501.6403,52.2165 ± 52.1376,2.9213 ± 0.9038,2.6281 ± 0.9104,20.5000 ± 5.7591
7,A7,"Quality-gated fusion, evidential loss without KL",0.9285 ± 0.0130,0.9331 ± 0.0113,0.9039 ± 0.0162,0.9331 ± 0.0113,0.9165 ± 0.0137,0.9288 ± 0.0131,0.9084 ± 0.0153,0.9092 ± 0.0146,0.0335 ± 0.0185,0.1193 ± 0.0227,0.2950 ± 0.0470,0.7619 ± 0.0512,756.4414 ± 143.3456,36.6255 ± 4.8049,2.8133 ± 0.7790,2.5330 ± 0.8105,20.9000 ± 4.4585
8,A8,Quality-gated fusion with softmax head,0.9120 ± 0.0099,0.9252 ± 0.0113,0.8788 ± 0.0167,0.9252 ± 0.0113,0.8960 ± 0.0116,0.9138 ± 0.0094,0.8878 ± 0.0113,0.8894 ± 0.0106,0.0437 ± 0.0133,0.1386 ± 0.0149,0.2936 ± 0.0412,0.8714 ± 0.0099,735.6189 ± 801.2499,58.9266 ± 74.8770,3.2453 ± 1.2530,2.9432 ± 1.3225,13.4000 ± 2.6750
9,A9,Full QGECAF: quality gate and evidential loss ...,0.9289 ± 0.0130,0.9352 ± 0.0094,0.9021 ± 0.0173,0.9352 ± 0.0094,0.9162 ± 0.0135,0.9293 ± 0.0129,0.9089 ± 0.0158,0.9098 ± 0.0152,0.0302 ± 0.0105,0.1165 ± 0.0179,0.2837 ± 0.0376,0.7625 ± 0.0416,1896.1901 ± 3454.9939,87.7776 ± 156.2502,2.7961 ± 0.6779,2.5160 ± 0.7127,20.8000 ± 3.6454



Mean, sample SD and 95% confidence intervals


,variant,model_variant,metric,folds,mean,sample_sd,ci95_low,ci95_high
0,A0,MobileNetV2 deep stream only,accuracy,10,0.928363,0.010209,0.921060,0.935666
1,A0,MobileNetV2 deep stream only,balanced_accuracy,10,0.935930,0.013801,0.926057,0.945802
2,A0,MobileNetV2 deep stream only,macro_precision,10,0.901829,0.013356,0.892275,0.911383
3,A0,MobileNetV2 deep stream only,macro_recall,10,0.935930,0.013801,0.926057,0.945802
4,A0,MobileNetV2 deep stream only,macro_f1,10,0.916536,0.011527,0.908291,0.924782
...,...,...,...,...,...,...,...,...
165,A9,Full QGECAF: quality gate and evidential loss ...,training_time_seconds,10,1896.190084,3454.993864,-575.363636,4367.743805
166,A9,Full QGECAF: quality gate and evidential loss ...,mean_epoch_time_seconds,10,87.777593,156.250218,-23.997079,199.552265
167,A9,Full QGECAF: quality gate and evidential loss ...,inference_time_seconds,10,2.796147,0.677931,2.311185,3.281110
168,A9,Full QGECAF: quality gate and evidential loss ...,inference_time_ms_per_image,10,2.515977,0.712688,2.006151,3.025803



Friedman test for macro F1


{'metric': 'macro_f1',
 'folds': 10,
 'models': ['A0', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9'],
 'friedman_statistic': 39.905454545454575,
 'friedman_p': 7.90414158120561e-06}


Paired Wilcoxon tests versus A9 with Holm correction


,reference,comparison,mean_difference_A9_minus_ablation,wilcoxon_statistic,raw_p,holm_adjusted_p,significant_at_0.05
0,A9,A0,-0.000309,24.0,0.769531,1.000000,False
1,A9,A1,0.079377,0.0,0.001953,0.017578,True
2,A9,A2,0.009586,9.0,0.064453,0.451172,False
3,A9,A3,0.002688,26.0,0.921875,1.000000,False
4,A9,A4,0.005676,17.0,0.322266,1.000000,False
5,A9,A5,0.003134,19.0,0.431641,1.000000,False
6,A9,A6,0.003000,19.0,0.431641,1.000000,False
7,A9,A7,-0.000272,23.0,0.695312,1.000000,False
8,A9,A8,0.020256,4.0,0.013672,0.109375,False


In [10]:
# ================================================================
# A0–A9 MODEL COMPLEXITY ANALYSIS
# Loads saved models only; DOES NOT retrain.
# ================================================================

import gc
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.python.framework.convert_to_constants import (
    convert_variables_to_constants_v2
)

warnings.filterwarnings("ignore")


# ----------------------------------------------------------------
# 1. CONFIGURATION
# ----------------------------------------------------------------

ABLATION_ROOT = Path(
    r"D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS"
)

OUTPUT_DIR = ABLATION_ROOT / "model_complexity"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# True: analyse all 100 saved models (10 variants × 10 folds).
# False: analyse only the first available fold of each variant.
ANALYSE_ALL_FOLDS = True

# Attempts actual GPU-memory measurement for batch size 1.
# Parameter and weight memory are always calculated.
MEASURE_GPU_MEMORY = True

EXPECTED_VARIANTS = [f"A{i}" for i in range(10)]

print("TensorFlow version:", tf.__version__)
print("Ablation root:", ABLATION_ROOT)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.10.0
Ablation root: D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [11]:
# ----------------------------------------------------------------
# 2. COLLECT CUSTOM KERAS OBJECTS
# ----------------------------------------------------------------

CUSTOM_OBJECT_NAMES = [
    "OneMinus",
    "EvidenceToDirichlet",
    "EvidentialLoss",
    "QualityGate",
    "QualityGatedFusion",
]

custom_objects = {}

for object_name in CUSTOM_OBJECT_NAMES:
    if object_name in globals():
        object_reference = globals()[object_name]

        custom_objects[object_name] = object_reference
        custom_objects[f"QGECAF>{object_name}"] = object_reference

print("Available custom objects:", sorted(custom_objects.keys()))

Available custom objects: []


In [13]:
# ================================================================
# QGECAF A0–A9 COMPLEXITY ANALYSIS FOR WEIGHTS-ONLY CHECKPOINTS
#
# Compatible with:
#   final_restored.weights.h5
#   best.weights.h5
#
# This code DOES NOT call model.fit() and DOES NOT retrain.
# ================================================================

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.python.framework.convert_to_constants import (
    convert_variables_to_constants_v2
)


# ----------------------------------------------------------------
# 1. CONFIGURATION
# ----------------------------------------------------------------

ABLATION_ROOT = Path(
    r"D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS"
)

OUTPUT_DIR = ABLATION_ROOT / "model_complexity"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VARIANTS = [f"A{i}" for i in range(10)]
NUM_CLASSES = 6
FAMILIES = ("wst", "gabor", "lbp", "hog", "shape")

# True loads the completed checkpoint to verify architecture compatibility.
# This still does not perform training.
LOAD_SAVED_WEIGHTS = True

MIB = 1024 ** 2


# ----------------------------------------------------------------
# 2. VERIFY REQUIRED NOTEBOOK OBJECTS
# ----------------------------------------------------------------

required_objects = [
    "cfg",
    "build_ablation_model",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run Sections 1–5 of the ablation notebook first.\n"
        f"Missing objects: {missing_objects}"
    )

if not ABLATION_ROOT.exists():
    raise FileNotFoundError(
        f"Ablation output folder does not exist:\n{ABLATION_ROOT}"
    )

print("TensorFlow:", tf.__version__)
print("Ablation root:", ABLATION_ROOT)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("\nNo model.fit() call is used in this analysis.")

TensorFlow: 2.10.0
Ablation root: D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

No model.fit() call is used in this analysis.


In [14]:
# ----------------------------------------------------------------
# 4. CHECK AVAILABLE FOLDS, PREPROCESSING FILES AND WEIGHTS
# ----------------------------------------------------------------

registry_rows = []

for fold in range(1, 11):
    fold_root = (
        ABLATION_ROOT /
        f"fold_{fold:02d}"
    )

    transformed_file = (
        fold_root /
        "shared_preprocessing" /
        "transformed_features.npz"
    )

    if not transformed_file.exists():
        print(
            f"Fold {fold}: preprocessing file not found; skipping."
        )
        continue

    with np.load(
        transformed_file,
        allow_pickle=False,
    ) as transformed:
        feature_dims = {
            family: int(
                transformed[family].shape[1]
            )
            for family in FAMILIES
        }

    for variant in VARIANTS:
        variant_dir = fold_root / variant

        final_weights = (
            variant_dir /
            "final_restored.weights.h5"
        )

        best_weights = (
            variant_dir /
            "best.weights.h5"
        )

        if final_weights.exists():
            weights_path = final_weights
            checkpoint_type = "final_restored"
        elif best_weights.exists():
            weights_path = best_weights
            checkpoint_type = "best"
        else:
            # Variant-fold is not yet complete or was not run.
            continue

        registry_rows.append({
            "variant": variant,
            "fold": fold,
            "weights_path": weights_path,
            "checkpoint_type": checkpoint_type,
            "feature_dims": feature_dims,
        })


registry_df = pd.DataFrame(registry_rows)

if registry_df.empty:
    raise FileNotFoundError(
        "No 'final_restored.weights.h5' or 'best.weights.h5' "
        f"files were found under:\n{ABLATION_ROOT}"
    )

registry_df["variant_number"] = (
    registry_df["variant"]
    .str.extract(r"(\d+)")
    .astype(int)
)

registry_df = registry_df.sort_values(
    ["variant_number", "fold"]
).drop(columns="variant_number").reset_index(drop=True)

print("Completed variant-fold checkpoints found:", len(registry_df))

availability = pd.crosstab(
    registry_df["variant"],
    registry_df["fold"],
)

display(availability)

Completed variant-fold checkpoints found: 100


fold,1,2,3,4,5,6,7,8,9,10
variant,,,,,,,,,,
A0,1,1,1,1,1,1,1,1,1,1
A1,1,1,1,1,1,1,1,1,1,1
A2,1,1,1,1,1,1,1,1,1,1
A3,1,1,1,1,1,1,1,1,1,1
A4,1,1,1,1,1,1,1,1,1,1
A5,1,1,1,1,1,1,1,1,1,1
A6,1,1,1,1,1,1,1,1,1,1
A7,1,1,1,1,1,1,1,1,1,1
A8,1,1,1,1,1,1,1,1,1,1


In [16]:
# ================================================================
# REQUIRED PARAMETER AND MEMORY FUNCTIONS
# Run this cell before the model-analysis loop.
# ================================================================

import numpy as np
import tensorflow as tf

MIB = 1024 ** 2


def shape_product(shape):
    dimensions = [
        int(value)
        for value in shape
        if value is not None
    ]

    return int(np.prod(dimensions)) if dimensions else 0


def dtype_bytes(dtype):
    return int(tf.as_dtype(dtype).size)


def variable_memory_bytes(variables):
    return int(sum(
        shape_product(variable.shape) *
        dtype_bytes(variable.dtype)
        for variable in variables
    ))


def parameter_statistics(model):
    trainable_parameters = int(sum(
        shape_product(variable.shape)
        for variable in model.trainable_weights
    ))

    nontrainable_parameters = int(sum(
        shape_product(variable.shape)
        for variable in model.non_trainable_weights
    ))

    total_parameters = (
        trainable_parameters +
        nontrainable_parameters
    )

    trainable_bytes = variable_memory_bytes(
        model.trainable_weights
    )

    nontrainable_bytes = variable_memory_bytes(
        model.non_trainable_weights
    )

    total_weight_bytes = (
        trainable_bytes +
        nontrainable_bytes
    )

    # Parameter-related Adam training memory:
    # weights + gradients + first moment + second moment.
    estimated_adam_bytes = (
        4 * trainable_bytes +
        nontrainable_bytes
    )

    return {
        "total_parameters":
            total_parameters,

        "trainable_parameters":
            trainable_parameters,

        "nontrainable_parameters":
            nontrainable_parameters,

        "total_parameters_million":
            total_parameters / 1e6,

        "trainable_parameters_million":
            trainable_parameters / 1e6,

        "nontrainable_parameters_million":
            nontrainable_parameters / 1e6,

        "trainable_parameter_percent": (
            100.0 * trainable_parameters /
            total_parameters
            if total_parameters else np.nan
        ),

        "trainable_weight_memory_mib":
            trainable_bytes / MIB,

        "nontrainable_weight_memory_mib":
            nontrainable_bytes / MIB,

        "total_weight_memory_mib":
            total_weight_bytes / MIB,

        "estimated_adam_parameter_memory_mib":
            estimated_adam_bytes / MIB,
    }


print("parameter_statistics() is now defined.")

parameter_statistics() is now defined.


In [21]:
# ----------------------------------------------------------------
# 5. REBUILD, LOAD AND ANALYSE EACH MODEL
# ----------------------------------------------------------------

complexity_rows = []
failure_rows = []

for row_index, row in registry_df.iterrows():
    variant = row["variant"]
    fold = int(row["fold"])
    feature_dims = row["feature_dims"]
    weights_path = Path(row["weights_path"])

    print("\n" + "=" * 72)
    print(
        f"[{row_index + 1}/{len(registry_df)}] "
        f"{variant}, fold {fold}"
    )
    print("Weights:", weights_path.name)
    print("Feature dimensions:", feature_dims)
    print("=" * 72)

    tf.keras.backend.clear_session()
    gc.collect()

    try:
        # Reconstruct the exact fold-specific model architecture.
        model = build_ablation_model(
            variant=variant,
            feature_dims=feature_dims,
            num_classes=NUM_CLASSES,
            cfg=cfg,
        )

        if LOAD_SAVED_WEIGHTS:
            model.load_weights(weights_path)
            weights_loaded = True
        else:
            weights_loaded = False

        parameter_result = parameter_statistics(
            model
        )

        try:
            flops = calculate_flops(model)
            gflops = flops / 1e9
            approximate_gmacs = flops / 2e9

        except Exception as flop_error:
            print(
                "FLOP calculation failed:",
                repr(flop_error),
            )

            flops = np.nan
            gflops = np.nan
            approximate_gmacs = np.nan

        checkpoint_size_mib = (
            weights_path.stat().st_size / MIB
        )

        complexity_rows.append({
            "variant": variant,
            "fold": fold,
            "checkpoint_type":
                row["checkpoint_type"],

            "weights_loaded":
                weights_loaded,

            "wst_dimension":
                feature_dims["wst"],

            "gabor_dimension":
                feature_dims["gabor"],

            "lbp_dimension":
                feature_dims["lbp"],

            "hog_dimension":
                feature_dims["hog"],

            "shape_dimension":
                feature_dims["shape"],

            "number_of_inputs":
                len(model.inputs),

            "checkpoint_size_mib":
                checkpoint_size_mib,

            "flops_batch_1":
                flops,

            "gflops_batch_1":
                gflops,

            "approximate_gmacs_batch_1":
                approximate_gmacs,

            **parameter_result,
        })

        print(
            "Total parameters:",
            f"{parameter_result['total_parameters']:,}",
        )

        print(
            "Trainable parameters:",
            f"{parameter_result['trainable_parameters']:,}",
        )

        print(
            "Non-trainable parameters:",
            f"{parameter_result['nontrainable_parameters']:,}",
        )

        print(
            "Weight memory:",
            f"{parameter_result['total_weight_memory_mib']:.2f} MiB",
        )

        print(
            "GFLOPs:",
            f"{gflops:.4f}",
        )

        print(
            "Approximate GMACs:",
            f"{approximate_gmacs:.4f}",
        )

    except Exception as error:
        print("FAILED:", repr(error))

        failure_rows.append({
            "variant": variant,
            "fold": fold,
            "weights_path": str(weights_path),
            "error": repr(error),
        })

    finally:
        if "model" in locals():
            del model

        tf.keras.backend.clear_session()
        gc.collect()


[1/100] A0, fold 1
Weights: final_restored.weights.h5
Feature dimensions: {'wst': 8, 'gabor': 7, 'lbp': 18, 'hog': 457, 'shape': 8}
Instructions for updating:
Use `tf.compat.v1.graph_util.tensor_shape_from_node_def_name`
Total parameters: 2,349,318
Trainable parameters: 1,617,734
Non-trainable parameters: 731,584
Weight memory: 8.96 MiB
GFLOPs: 0.1983
Approximate GMACs: 0.0991

[2/100] A0, fold 2
Weights: final_restored.weights.h5
Feature dimensions: {'wst': 8, 'gabor': 7, 'lbp': 17, 'hog': 463, 'shape': 7}
Total parameters: 2,349,318
Trainable parameters: 1,617,734
Non-trainable parameters: 731,584
Weight memory: 8.96 MiB
GFLOPs: 0.1983
Approximate GMACs: 0.0991

[3/100] A0, fold 3
Weights: final_restored.weights.h5
Feature dimensions: {'wst': 8, 'gabor': 7, 'lbp': 19, 'hog': 464, 'shape': 8}
Total parameters: 2,349,318
Trainable parameters: 1,617,734
Non-trainable parameters: 731,584
Weight memory: 8.96 MiB
GFLOPs: 0.1983
Approximate GMACs: 0.0991

[4/100] A0, fold 4
Weights: final_

In [18]:
# ================================================================
# FLOP AND GMAC CALCULATION FOR TENSORFLOW 2.10
# Run before rerunning the model-analysis loop.
# ================================================================

import numpy as np
import tensorflow as tf

from tensorflow.python.framework.convert_to_constants import (
    convert_variables_to_constants_v2
)


def create_input_specs(model):
    """
    Create batch-size-1 TensorSpecs in the exact order expected by
    the Keras model.
    """
    input_specs = []

    for input_tensor in model.inputs:
        input_shape = [1]

        for dimension in input_tensor.shape[1:]:
            if dimension is None:
                input_shape.append(1)
            else:
                input_shape.append(int(dimension))

        input_specs.append(
            tf.TensorSpec(
                shape=input_shape,
                dtype=input_tensor.dtype,
                name=input_tensor.name.split(":")[0],
            )
        )

    return input_specs


def calculate_flops(model):
    """
    Calculate approximate FLOPs for one forward pass with batch size 1.

    Returns:
        Total floating-point operations as a numeric value.
    """
    input_specs = create_input_specs(model)

    if len(input_specs) == 1:

        @tf.function
        def forward_function(input_0):
            return model(
                input_0,
                training=False,
            )

    else:

        @tf.function
        def forward_function(*model_inputs):
            return model(
                list(model_inputs),
                training=False,
            )

    concrete_function = (
        forward_function.get_concrete_function(
            *input_specs
        )
    )

    frozen_function = (
        convert_variables_to_constants_v2(
            concrete_function
        )
    )

    frozen_graph_definition = (
        frozen_function.graph.as_graph_def()
    )

    with tf.Graph().as_default() as profiling_graph:
        tf.compat.v1.graph_util.import_graph_def(
            frozen_graph_definition,
            name="",
        )

        profiling_options = (
            tf.compat.v1.profiler
            .ProfileOptionBuilder
            .float_operation()
        )

        # Prevent TensorFlow from printing the complete layer-by-layer
        # profiler report.
        profiling_options["output"] = "none"

        profile_result = (
            tf.compat.v1.profiler.profile(
                graph=profiling_graph,
                cmd="op",
                options=profiling_options,
            )
        )

    if profile_result is None:
        raise RuntimeError(
            "TensorFlow FLOP profiler returned no result."
        )

    return float(
        profile_result.total_float_ops
    )


print("calculate_flops() has been defined successfully.")

calculate_flops() has been defined successfully.


In [19]:
required_functions = [
    "build_ablation_model",
    "parameter_statistics",
    "calculate_flops",
    "create_input_specs",
]

missing_functions = [
    name for name in required_functions
    if name not in globals()
]

if missing_functions:
    raise RuntimeError(
        "The following functions are missing: "
        f"{missing_functions}"
    )

print("PASS: All required complexity functions are available.")

PASS: All required complexity functions are available.


In [20]:
# model is available only if the previous analysis was interrupted
# while the model object remained in memory.

if "model" in globals():
    test_flops = calculate_flops(model)

    print("FLOPs:", f"{test_flops:,.0f}")
    print("GFLOPs:", f"{test_flops / 1e9:.4f}")
    print(
        "Approximate GMACs:",
        f"{test_flops / 2e9:.4f}",
    )
else:
    print(
        "No model object is currently available. "
        "Proceed directly to rerun the analysis loop."
    )

No model object is currently available. Proceed directly to rerun the analysis loop.


In [22]:
# ----------------------------------------------------------------
# 6. SAVE FOLD-WISE RESULTS
# ----------------------------------------------------------------

complexity_df = pd.DataFrame(
    complexity_rows
)

if complexity_df.empty:
    raise RuntimeError(
        "No models were successfully analysed. Examine the "
        "failure messages printed above."
    )

complexity_df["variant_number"] = (
    complexity_df["variant"]
    .str.extract(r"(\d+)")
    .astype(int)
)

complexity_df = complexity_df.sort_values(
    ["variant_number", "fold"]
).drop(columns="variant_number")

foldwise_file = (
    OUTPUT_DIR /
    "ablation_model_complexity_foldwise.csv"
)

complexity_df.to_csv(
    foldwise_file,
    index=False,
)

if failure_rows:
    pd.DataFrame(failure_rows).to_csv(
        OUTPUT_DIR /
        "ablation_model_complexity_failures.csv",
        index=False,
    )

display(complexity_df.round(4))

print("Saved fold-wise results to:")
print(foldwise_file)

,variant,fold,checkpoint_type,weights_loaded,wst_dimension,gabor_dimension,lbp_dimension,hog_dimension,shape_dimension,number_of_inputs,...,trainable_parameters,nontrainable_parameters,total_parameters_million,trainable_parameters_million,nontrainable_parameters_million,trainable_parameter_percent,trainable_weight_memory_mib,nontrainable_weight_memory_mib,total_weight_memory_mib,estimated_adam_parameter_memory_mib
0,A0,1,final_restored,True,8,7,18,457,8,1,...,1617734,731584,2.3493,1.6177,0.7316,68.8597,6.1712,2.7908,8.9619,27.4754
1,A0,2,final_restored,True,8,7,17,463,7,1,...,1617734,731584,2.3493,1.6177,0.7316,68.8597,6.1712,2.7908,8.9619,27.4754
2,A0,3,final_restored,True,8,7,19,464,8,1,...,1617734,731584,2.3493,1.6177,0.7316,68.8597,6.1712,2.7908,8.9619,27.4754
3,A0,4,final_restored,True,8,7,17,471,7,1,...,1617734,731584,2.3493,1.6177,0.7316,68.8597,6.1712,2.7908,8.9619,27.4754
4,A0,5,final_restored,True,8,7,19,458,9,1,...,1617734,731584,2.3493,1.6177,0.7316,68.8597,6.1712,2.7908,8.9619,27.4754
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,A9,6,final_restored,True,8,7,18,458,9,7,...,1684359,731584,2.4159,1.6844,0.7316,69.7185,6.4253,2.7908,9.2161,28.4921
96,A9,7,final_restored,True,8,6,18,450,8,7,...,1683719,731584,2.4153,1.6837,0.7316,69.7105,6.4229,2.7908,9.2136,28.4823
97,A9,8,final_restored,True,8,7,20,482,8,7,...,1685959,731584,2.4175,1.6860,0.7316,69.7385,6.4314,2.7908,9.2222,28.5165
98,A9,9,final_restored,True,8,7,18,466,9,7,...,1684871,731584,2.4165,1.6849,0.7316,69.7249,6.4273,2.7908,9.2180,28.4999


Saved fold-wise results to:
D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS\model_complexity\ablation_model_complexity_foldwise.csv


In [23]:
# ----------------------------------------------------------------
# 7. CREATE A0–A9 SUMMARY
# ----------------------------------------------------------------

summary_metrics = [
    "total_parameters",
    "trainable_parameters",
    "nontrainable_parameters",
    "total_parameters_million",
    "trainable_parameters_million",
    "nontrainable_parameters_million",
    "trainable_parameter_percent",
    "total_weight_memory_mib",
    "estimated_adam_parameter_memory_mib",
    "checkpoint_size_mib",
    "gflops_batch_1",
    "approximate_gmacs_batch_1",
]

summary_rows = []

for variant, group in complexity_df.groupby(
    "variant"
):
    summary_row = {
        "variant": variant,
        "available_folds": len(group),
    }

    for metric in summary_metrics:
        values = pd.to_numeric(
            group[metric],
            errors="coerce",
        ).dropna()

        summary_row[
            f"{metric}_mean"
        ] = (
            values.mean()
            if len(values) else np.nan
        )

        summary_row[
            f"{metric}_sample_SD"
        ] = (
            values.std(ddof=1)
            if len(values) > 1
            else np.nan
        )

    summary_rows.append(summary_row)


summary_df = pd.DataFrame(
    summary_rows
)

summary_df["variant_number"] = (
    summary_df["variant"]
    .str.extract(r"(\d+)")
    .astype(int)
)

summary_df = summary_df.sort_values(
    "variant_number"
).drop(columns="variant_number")

summary_file = (
    OUTPUT_DIR /
    "ablation_model_complexity_summary.csv"
)

summary_df.to_csv(
    summary_file,
    index=False,
)

display(summary_df.round(4))

print("Saved summary to:")
print(summary_file)

,variant,available_folds,total_parameters_mean,total_parameters_sample_SD,trainable_parameters_mean,trainable_parameters_sample_SD,nontrainable_parameters_mean,nontrainable_parameters_sample_SD,total_parameters_million_mean,total_parameters_million_sample_SD,...,total_weight_memory_mib_mean,total_weight_memory_mib_sample_SD,estimated_adam_parameter_memory_mib_mean,estimated_adam_parameter_memory_mib_sample_SD,checkpoint_size_mib_mean,checkpoint_size_mib_sample_SD,gflops_batch_1_mean,gflops_batch_1_sample_SD,approximate_gmacs_batch_1_mean,approximate_gmacs_batch_1_sample_SD
0,A0,10,2349318.0,0.0000,1617734.0,0.0000,731584.0,0.0,2.3493,0.0000,...,8.9619,0.0000,27.4754,0.0000,9.1900,0.0000,0.1983,0.0,0.0991,0.0
1,A1,10,42361.2,674.4843,42361.2,674.4843,0.0,0.0,0.0424,0.0007,...,0.1616,0.0026,0.6464,0.0103,0.2166,0.0026,0.0001,0.0,0.0000,0.0
2,A2,10,2390841.2,674.4843,1659257.2,674.4843,731584.0,0.0,2.3908,0.0007,...,9.1203,0.0026,28.1090,0.0103,9.3939,0.0026,0.1984,0.0,0.0992,0.0
3,A3,10,2399225.2,674.4843,1667641.2,674.4843,731584.0,0.0,2.3992,0.0007,...,9.1523,0.0026,28.2370,0.0103,9.4296,0.0025,0.1983,0.0,0.0992,0.0
4,A4,10,2399225.2,674.4843,1667641.2,674.4843,731584.0,0.0,2.3992,0.0007,...,9.1523,0.0026,28.2370,0.0103,9.4296,0.0025,0.1983,0.0,0.0992,0.0
5,A5,10,2415993.2,674.4843,1684409.2,674.4843,731584.0,0.0,2.4160,0.0007,...,9.2163,0.0026,28.4928,0.0103,9.5075,0.0026,0.1984,0.0,0.0992,0.0
6,A6,10,2415993.2,674.4843,1684409.2,674.4843,731584.0,0.0,2.4160,0.0007,...,9.2163,0.0026,28.4928,0.0103,9.5091,0.0026,0.1984,0.0,0.0992,0.0
7,A7,10,2416122.2,674.4843,1684538.2,674.4843,731584.0,0.0,2.4161,0.0007,...,9.2168,0.0026,28.4948,0.0103,9.5165,0.0026,0.1984,0.0,0.0992,0.0
8,A8,10,2416122.2,674.4843,1684538.2,674.4843,731584.0,0.0,2.4161,0.0007,...,9.2168,0.0026,28.4948,0.0103,9.5154,0.0026,0.1984,0.0,0.0992,0.0
9,A9,10,2416122.2,674.4843,1684538.2,674.4843,731584.0,0.0,2.4161,0.0007,...,9.2168,0.0026,28.4948,0.0103,9.5165,0.0026,0.1984,0.0,0.0992,0.0


Saved summary to:
D:\chaitali\qgacef\QGECAF_ABLATION_SAME_LOCKED_PATIENTS\model_complexity\ablation_model_complexity_summary.csv
